[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap06/cap06_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 6 Analyse de documents et inspection industrielle

Dans la **Partie I — Traitement numérique des images (TNI)**, ont été étudiées des techniques de transformation et d’amélioration des images, telles que les opérations morphologiques, le filtrage spatial, les convolutions, le seuillage, la segmentation et le traitement dans le domaine fréquentiel.

La **Partie II — Vision par ordinateur (VO)** élargit ce champ en abordant l’interprétation automatique du contenu visuel, impliquant l’extraction d’informations, la reconnaissance de formes et la prise de décisions à partir d’images.

Ce chapitre présente cette transition à travers deux applications représentatives :

1. **Analyse automatisée de documents**, appliquée au traitement de formulaires, d’évaluations et d’autres documents structurés au moyen de systèmes de reconnaissance optique de marques (*Optical Mark Recognition* – OMR) ;
2. **Inspection industrielle automatisée**, axée sur le contrôle qualité et la détection de défauts sur les chaînes de production.

Ces applications intègrent des techniques de détection de structures géométriques, d’extraction de descripteurs invariants, de reconnaissance de formes et de classification d’objets, constituant la base de nombreux systèmes modernes d’inspection visuelle et d’automatisation.

## 6.1 Objectifs du chapitre

À la fin de ce chapitre, l'étudiant devrait être capable de :

* **Évaluer l'influence du prétraitement** sur la qualité de la reconnaissance automatique des informations dans les documents ;
* **Effectuer la reconnaissance optique de caractères** (ROC) pour convertir des documents numérisés en texte codé ;
* **Appliquer des techniques de traitement du langage naturel**, y compris la traduction automatique, au texte obtenu par ROC ;
* Appliquer des techniques d'**alignement et de rectification géométrique de documents** utilisant la transformée de Hough et les transformations projectives ;
* **Implémenter des systèmes de reconnaissance optique de marques** (ROM) pour la lecture automatisée d'évaluations et de formulaires ;
* **Détecter et segmenter des régions d'intérêt** sur la base d'opérations morphologiques, de contours et de propriétés géométriques ;
* **Décoder des marqueurs bidimensionnels et des codes-barres**, en intégrant des bibliothèques de vision par ordinateur aux *pipelines* de traitement documentaire ;
* **Développer des *pipelines* de vision par ordinateur** pour l'analyse automatisée de documents.

Ce chapitre marque la transition du **traitement numérique des images**, axé sur la transformation des images, vers la **vision par ordinateur**, dont l'objectif est d'interpréter le contenu visuel, d'extraire des informations et de soutenir des processus automatisés d'analyse et de prise de décision.

## 6.2 Configuration de l'environnement

Les exemples de ce chapitre utilisent des bibliothèques largement employées en
PDI-VC. Le bloc ci-dessous
installe les paquets nécessaires ; dans les environnements qui les possèdent déjà, l'exécution
peut être ignorée.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm

# installer plus de dépendances en plus de morph.py pour ce chapitre
import sys, subprocess, importlib, shutil

def setup_cap06():
    """Installe les dépendances système et Python spécifiques au chapitre 6
    (OCR, lecture de PDF, code-barres)."""

    # 1. Dépendances système
    if 'google.colab' in sys.modules:
        print("[ENVIRONNEMENT] Google Colab. Configuration des dépendances système...")
        subprocess.run(
            "apt-get update && apt-get install -y poppler-utils "
            "libzbar0 tesseract-ocr tesseract-ocr-por",
            shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    elif shutil.which("tesseract") is None:
        # Environnement local sans tesseract : tentative d'installation via apt-get (nécessite sudo/root)
        if shutil.which("apt-get"):
            print("[ENVIRONNEMENT] Local. Installation de tesseract-ocr via apt-get (peut demander un mot de passe)...")
            resultado = subprocess.run(
                "sudo apt-get update && sudo apt-get install -y tesseract-ocr tesseract-ocr-por",
                shell=True
            )
            if resultado.returncode != 0 or shutil.which("tesseract") is None:
                print(
                    "[AVERTISSEMENT] Impossible d'installer automatiquement. "
                    "Installez manuellement : sudo apt install tesseract-ocr tesseract-ocr-por"
                )
        else:
            print(
                "[AVERTISSEMENT] tesseract introuvable et apt-get indisponible. "
                "Installez manuellement avant d'exécuter les cellules OCR."
            )

    # 2. Dépendances Python (installe uniquement celles manquantes)
    pkgs = {
        "cv2": "opencv-python", "skimage": "scikit-image", "numpy": "numpy",
        "pdf2image": "pdf2image", "pandas": "pandas", "tabulate": "tabulate",
        "PyPDF2": "PyPDF2", "bcrypt": "bcrypt", "pyarrow": "pyarrow",
        "pyzbar": "pyzbar", "pytesseract": "pytesseract", "deep_translator": "deep-translator"
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            resultado_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
            if resultado_pip.returncode != 0:
                print(f"[AVERTISSEMENT] Échec de l'installation de {pkg} (nécessaire pour le module {mod}).")


setup_cap06()

# 3. Imports globaux du pipeline
import cv2, numpy as np, matplotlib.pyplot as plt
from skimage import io, data, color

✅ Environnement prêt. Morph : 1.1.9 | OpenCV : 5.0.0
[ENVIRONNEMENT] Local. Installation de tesseract-ocr via apt-get (peut demander un mot de passe)...
[AVERTISSEMENT] Impossible d'installer automatiquement. Installez manuellement : sudo apt install tesseract-ocr tesseract-ocr-por


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: uma senha é necessária


Outre ces bibliothèques, on utilisera le module didactique `morph.py`,
développé pour simplifier les opérations de lecture, de visualisation et de
traitement d'images tout au long de cet ouvrage. Le code suivant vérifie
sa disponibilité, effectue le *téléchargement* si nécessaire et confirme la
version chargée.

In [2]:
import os
import urllib.request

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/morph.py",
        "morph.py",
    )

import morph
from morph import mm

print(f"✅ Environnement prêt. morph {getattr(morph, '__version__', 'local_file')}")

✅ Environnement prêt. morph 1.1.9


## 6.3 Bases d’images pour l’expérimentation

Les exemples présentés dans cette partie de l’ouvrage utilisent, chaque fois que possible, des documents numérisés, des feuilles de réponses, des codes-barres, des *QRCodes* et d’autres images provenant d’applications réelles. Afin de rendre les expériences entièrement reproductibles — y compris dans des environnements sans accès à Internet ou aux fichiers originaux du MCTest —, on emploie également des images publiques largement utilisées dans l’enseignement et la recherche en Traitement Numérique des Images et Vision par Ordinateur (TNI-VO).

> ### 💡 Pourquoi utiliser une base d’images de *benchmark* ?
>
> Des images comme `camera()` et `coins()` sont employées depuis des décennies dans les manuels, les articles scientifiques et les supports pédagogiques de TNI-VO. Leur utilisation offre d’importants avantages :
>
> - **Reproductibilité :** tout lecteur obtient exactement les mêmes images, quel que soit l’ordinateur ou le système d’exploitation utilisé, sans avoir besoin de *téléchargements* externes ni de fichiers spécifiques à cet ouvrage ;
> - **Comparabilité :** les résultats produits peuvent être directement comparés à ceux rapportés dans la littérature, puisque les mêmes ensembles d’images sont largement adoptés comme référence ;
> - **Concentration sur les algorithmes :** étant compactes, bien documentées et distribuées sans restrictions d’usage à des fins éducatives et scientifiques, ces images permettent de concentrer l’attention sur les techniques de traitement, en réduisant les interférences liées à l’acquisition ou à la gestion des données.

### 6.3.1 Images publiques avec `skimage.data`

Le module `skimage.data` fournit une collection d'images de référence largement utilisée dans les activités d'enseignement, de recherche et de validation d'algorithmes en TDI-VC. L'inspection de l'attribut `data.__all__` montre que la version actuelle regroupe **42 éléments**, incluant des photographies naturelles, des documents numérisés, des images médicales, de la microscopie, des textures, des motifs synthétiques, des modèles tridimensionnels et des séquences temporelles. Il convient de noter que certains de ces éléments correspondent à des fonctions utilitaires, comme `data_dir()` et `download_all()`, et non à des images à proprement parler.

Comme l'objectif de ce chapitre est de présenter des applications d'analyse documentaire et d'inspection visuelle, la [Tableau 6.2](#tbl-06-skimage-data) regroupe une **sélection représentative** des images les plus pertinentes, organisée selon leurs principales applications en Vision par Ordinateur. La [Figure 6.1](#fig-06-skimage-data) présente un échantillon de ces images, groupées selon la même classification adoptée dans le tableau.

> ### 📝 Images utilisées dans ce chapitre
>
> Bien que `skimage.data` fournisse des dizaines d'images de référence, seulement quatre sont employées directement dans les expériences de ce chapitre. Elles ont été sélectionnées car elles reproduisent, de manière contrôlée, des caractéristiques fréquemment rencontrées dans les documents numérisés et dans les systèmes d'inspection visuelle industrielle. La [Tableau 6.1](#tbl-06-skimage-cap06) résume le rôle de chacune d'elles au long de ce chapitre.
>
> | Image | Application dans le chapitre |
> |---|---|
> | `data.page()` | Page numérisée utilisée dans les expériences de correction d'éclairage, d'amélioration locale (CLAHE) et de seuillage automatique par Otsu. |
> | `data.text()` | Document contenant du texte imprimé, employé pour illustrer la segmentation, l'extraction de contours et les étapes typiques de l'OCR et de l'OMR. |
> | `data.coffee()` | Photographie en couleurs avec des variations naturelles d'éclairage, utilisée pour exemplifier des techniques applicables à des scènes réelles non documentaires. |
> | `data.brick()` | Texture de référence employée dans des exemples d'inspection de surface et de détection de défauts par analyse de variance locale. |
>
> : Images de `skimage.data` utilisées dans les expériences de ce chapitre. {#tbl-06-skimage-cap06}

In [3]:
# @title { display-mode: "form" }
import pandas as pd
from IPython.display import Markdown

# Chaque catégorie est associée à une couleur, réutilisée dans le titre des images de la @fig-06-skimage-data
categorias = {
    "📄 Documentos & OCR/OMR": {
        "cor": "#2563eb",
        "itens": [
            ("`data.page()`", "Página digitalizada de documento — normalização de fundo, CLAHE e Otsu."),
            ("`data.text()`", "Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR."),
        ],
    },
    "🔵 Segmentação, Morfologia & Contornos": {
        "cor": "#16a34a",
        "itens": [
            ("`data.coins()`", "Conjunto de moedas — referência clássica para segmentação e *watershed*."),
            ("`data.clock()`", "Relógio analógico — detecção de formas e contornos."),
            ("`data.binary_blobs()`", "Blobs binários sintéticos — conectividade e morfologia matemática."),
            ("`data.moon()`", "Superfície lunar — segmentação de crateras por relevo de intensidade."),
        ],
    },
    "🧵 Textura & Inspeção Industrial": {
        "cor": "#ea580c",
        "itens": [
            ("`data.brick()`", "Textura uniforme de tijolos — detecção de defeitos por variância local."),
            ("`data.checkerboard()`", "Padrão xadrez — calibração de câmera e transformações geométricas."),
        ],
    },
    "🖼️ Fotografias Clássicas de PDI/VC": {
        "cor": "#7c3aed",
        "itens": [
            ("`data.camera()`", "Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI."),
            ("`data.astronaut()`", "Retrato colorido de astronauta — filtragem e realce em cor."),
            ("`data.coffee()`", "Xícara de café — cena real com variação de iluminação e cor."),
            ("`data.cat()` / `data.chelsea()`", "Fotografias coloridas de gatos — detecção de bordas e realce."),
            ("`data.horse()`", "Silhueta binária de cavalo — descritores de forma e contorno."),
        ],
    },
}

linhas = []
for cat, info in categorias.items():
    for funcao, desc in info["itens"]:
        linhas.append({"Categoria": cat, "Função": funcao, "Descrição": desc})

df = pd.DataFrame(linhas)
Markdown(df.to_markdown(index=False, colalign=("left", "left", "left")))


**Tableau 6.2:** Sélection d’images publiques représentatives disponibles dans le module *skimage.data*, organisées par domaine d’application.


| Categoria                              | Função                          | Descrição                                                                    |
|:---------------------------------------|:--------------------------------|:-----------------------------------------------------------------------------|
| 📄 Documentos & OCR/OMR                | `data.page()`                   | Página digitalizada de documento — normalização de fundo, CLAHE e Otsu.      |
| 📄 Documentos & OCR/OMR                | `data.text()`                   | Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR. |
| 🔵 Segmentação, Morfologia & Contornos | `data.coins()`                  | Conjunto de moedas — referência clássica para segmentação e *watershed*.     |
| 🔵 Segmentação, Morfologia & Contornos | `data.clock()`                  | Relógio analógico — detecção de formas e contornos.                          |
| 🔵 Segmentação, Morfologia & Contornos | `data.binary_blobs()`           | Blobs binários sintéticos — conectividade e morfologia matemática.           |
| 🔵 Segmentação, Morfologia & Contornos | `data.moon()`                   | Superfície lunar — segmentação de crateras por relevo de intensidade.        |
| 🧵 Textura & Inspeção Industrial       | `data.brick()`                  | Textura uniforme de tijolos — detecção de defeitos por variância local.      |
| 🧵 Textura & Inspeção Industrial       | `data.checkerboard()`           | Padrão xadrez — calibração de câmera e transformações geométricas.           |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.camera()`                 | Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI. |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.astronaut()`              | Retrato colorido de astronauta — filtragem e realce em cor.                  |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.coffee()`                 | Xícara de café — cena real com variação de iluminação e cor.                 |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.cat()` / `data.chelsea()` | Fotografias coloridas de gatos — detecção de bordas e realce.                |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.horse()`                  | Silhueta binária de cavalo — descritores de forma e contorno.                |

In [4]:
import matplotlib.pyplot as plt
from skimage import data

# Images triées par catégorie ; la couleur du titre reproduit la couleur de la catégorie dans le tableau précédent
imgs = [
    ("page",         data.page(),         "#2563eb"),   # Documents & OCR/OMR
    ("text",         data.text(),         "#2563eb"),
    ("coins",        data.coins(),        "#16a34a"),   # Segmentation & morphologie
    ("binary_blobs", data.binary_blobs(), "#16a34a"),
    ("brick",        data.brick(),        "#ea580c"),   # Texture & inspection industrielle
    ("checkerboard", data.checkerboard(),"#ea580c"),
    ("camera",       data.camera(),       "#7c3aed"),   # Photographies classiques de TNI/Vision par ordinateur
    ("coffee",       data.coffee(),       "#7c3aed"),
]

fig, ax = plt.subplots(2, 4, figsize=(11, 5.5))
for a, (nome, img, cor) in zip(ax.ravel(), imgs):
    a.imshow(img, cmap="gray")
    a.set_title(nome, color=cor, fontweight="bold")
    a.axis("off")
plt.tight_layout()


<Figure size 3300x1650 with 8 Axes>

**Figure 6.1:** Échantillon d


## 6.4 Normalisation du fond et égalisation locale de contraste

La qualité de la segmentation dépend directement des caractéristiques de l'image d'entrée. Dans les documents numérisés, les variations d'éclairage, les ombres, les régions surexposées et les différences de tonalité du papier réduisent le contraste entre le premier plan et le fond, rendant difficile l'application de méthodes de seuillage global, comme l'algorithme d'Otsu.

Pour minimiser ces effets, deux techniques complémentaires de prétraitement sont employées :

- **Normalisation du fond :** elle estime la composante basse fréquence de l'image au moyen d'un fort lissage, puis normalise l'image originale par rapport à ce fond estimé. Cette procédure réduit les gradients d'éclairage et compense les variations lentes d'intensité, tout en préservant les structures d'intérêt.
- **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*), présenté au **Chapitre 4 :** il divise l'image en petites régions (*tiles*) et procède à l'égalisation de l'histogramme de chaque région de manière indépendante. Le contraste est limité afin d'éviter l'amplification excessive du bruit, ce qui rend la technique particulièrement adaptée aux images présentant des variations locales d'éclairage.

La [Figure 6.2](#fig-06-clahe-page) compare ces stratégies en utilisant l'image `page()` de la bibliothèque `skimage.data`. Six résultats sont présentés : (a) l'image originale ; (b) la binarisation directe par la méthode d'Otsu, utilisée comme référence ; (c) le fond estimé par filtrage gaussien ; (d) l'image après la normalisation du fond ; (e) la binarisation obtenue après application du CLAHE suivie de la méthode d'Otsu ; et (f) la binarisation obtenue après normalisation du fond suivie de l'application de la méthode d'Otsu.

La comparaison permet d'observer l'effet produit par chaque étape du prétraitement et son influence sur la qualité de la segmentation. En particulier, la normalisation du fond réduit les variations globales d'éclairage, tandis que le CLAHE augmente le contraste local entre les caractères et le fond. Selon les caractéristiques de l'image, l'une ou l'autre stratégie peut produire des résultats supérieurs, aucune technique n'étant universellement plus adaptée.

In [5]:
import cv2
from skimage import data
from morph import mm

img = data.page()  # ou : img = mm.gray(img_final) — avec l'image de la feuille d'examen

# ── Méthode 1 : Normalisation du fond + Otsu ──────────────────────────────
# Estime le fond avec un filtre gaussien à grand sigma (variations lentes de lumière)
# et divise pixel par pixel pour annuler le gradient d'éclairage
bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)
img_norm_otsu = mm.threshold(img_norm)

# ── Méthode 2 : CLAHE + Otsu ──────────────────────────────────────────────
# tileGridSize définit la taille de chaque région locale (tuile)
# clipLimit contrôle le plafond d'amplification — des valeurs élevées augmentent le contraste
# mais amplifient aussi le bruit
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Méthode 0 : Otsu direct (sans prétraitement) — référence ───────────
img_otsu = mm.threshold(img)

mm.show(
    [img,  img_otsu, bg, img_norm, img_clahe_otsu, img_norm_otsu],
    titles=["(a) Originale", "(b) Otsu direct", "(c) Gaussien",  \
            "(d) Fond normalisé (img/bg)", "(e) CLAHE + Otsu", \
            "(f) Normalisation du fond + Otsu"],
    cols=3,
    figsize=(14, 8)
)


<Figure size 2100x1200 with 6 Axes>

**Figure 6.2:** Comparaison entre stratégies de prétraitement pour la binarisation de l


> ### 📝 🧠 Pourquoi cela fonctionne-t-il ? — Normalisation du fond vs CLAHE
>
> **Normalisation du fond :** en divisant l’image par une version fortement lissée
> d’elle-même, on élimine les variations lentes de luminosité (gradient de lumière,
> ombre de bord) sans affecter les détails fins — texte, lignes, bulles.
> Le résultat est une image à l’éclairage approximativement uniforme, où le seuil
> global d’Otsu fonctionne désormais correctement sur toute la page.
>
> **CLAHE :** un histogramme global égalisé « étire » les tons de toute l’image d’un
> coup — utile lorsque l’éclairage est uniforme, mais problématique lorsqu’il ne l’est pas.
> Le CLAHE divise l’image en petits blocs (*tiles*) et égalise chacun
> séparément, avec une limite maximale d’amplification (`clipLimit`) pour ne pas exploser
> le bruit. Il est particulièrement efficace pour rehausser les régions sous-exposées localement,
> mais il n’élimine pas les gradients globaux — c’est pourquoi l’appliquer après la normalisation du
> fond tend à produire des résultats plus cohérents.

## 6.5 Reconnaissance Optique de Caractères (OCR)

Après la binarisation, l'étape suivante du traitement documentaire consiste à convertir la représentation visuelle des caractères en texte codé numériquement, processus appelé **Reconnaissance Optique de Caractères** (*Optical Character Recognition* — **OCR**).

De manière générale, un système d'OCR comprend trois étapes :

1. **Segmentation :** identifie les lignes, les mots et les caractères dans l'image, en utilisant des projections horizontales et verticales ou la détection de composantes connexes.
2. **Extraction de caractéristiques :** représente chaque caractère par des attributs visuels, tels que les bords, les courbures et les motifs de tracé.
3. **Reconnaissance :** associe les attributs extraits au caractère le plus probable. Les systèmes actuels utilisent principalement des réseaux neuronaux récurrents (LSTM) ou des architectures basées sur les *transformers*.

Dans cet ouvrage, on utilise le **Tesseract OCR**, accessible via la bibliothèque `pytesseract` (installation : `pip install pytesseract`). Développé à l'origine par Hewlett-Packard entre 1985 et 1995 et actuellement maintenu par Google, Tesseract est décrit dans Smith (2007) et Smith (2013). Dans les versions récentes, la reconnaissance textuelle est réalisée par des réseaux neuronaux LSTM.

Les performances de l'OCR dépendent de la qualité de l'image d'entrée. Le bruit, le faible contraste, les distorsions géométriques et l'éclairage irrégulier réduisent le taux de reconnaissance. Pour cette raison, des étapes telles que la rectification, la normalisation du fond et l'égalisation adaptative (CLAHE) intègrent le prétraitement de l'image.

> ### 📝 🧠 Le Tesseract a-t-il besoin d'une image binarisée ?
>
> Le Tesseract intègre en interne une étape de binarisation adaptative avant la reconnaissance des caractères. Pour cette raison, fournir à l'OCR une image préalablement binarisée ne produit pas toujours les meilleurs résultats.
>
> Comme le seuillage est une opération irréversible, il peut éliminer les variations subtiles d'intensité sur les bords des caractères, comme l'*anticrénelage*, qui peuvent aider le mécanisme de reconnaissance. Dans de nombreux cas, une image en niveaux de gris, avec un bon éclairage et un bon contraste, produit une transcription plus fidèle que sa version binarisée.

Pour étudier cet effet, on compare le texte extrait par Tesseract à partir de quatre versions de la même image, présentées dans la [Figure 6.3](#fig-06-ocr-comparacao): (a) image originale ; (b) image soumise à l'égalisation adaptative (CLAHE) suivie d'un seuillage par la méthode d'Otsu ; (c) image soumise à la normalisation du fond suivie d'un seuillage par la méthode d'Otsu ; et (d) image soumise uniquement à la normalisation du fond, en préservant les niveaux de gris.

La comparaison entre les versions (c) et (d) montre que, dans les passages contenant des caractères visuellement similaires, la version en niveaux de gris (d) a produit une transcription plus fidèle au texte original que la version binarisée (c). Ce résultat indique que le seuillage appliqué lors du prétraitement peut éliminer des informations utiles à la reconnaissance. Ainsi, bien que la binarisation soit essentielle pour diverses opérations de traitement d'images, elle ne constitue pas nécessairement la meilleure entrée pour l'OCR. Le choix de la technique de prétraitement doit tenir compte de l'étape suivante du *pipeline* documentaire.

In [6]:
import pytesseract
import shutil as _sh
if _sh.which("tesseract") is None:
    # Environnement de build sans Tesseract installé (ex. : sans apt/sudo) :
    # dégrade au lieu de casser le rendu. Sur Colab/local avec Tesseract,
    # rien ne change.
    _AVISO_OCR = "[Tesseract OCR indisponivel neste ambiente - texto omitido]"
    pytesseract.image_to_string = lambda *a, **k: _AVISO_OCR
from skimage import data
import cv2
from morph import mm

img = data.page()

# ── Réutilisation des résultats de la section précédente ───────────────────
img_otsu = mm.threshold(img)

bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)       # niveaux de gris, sans Otsu
img_norm_otsu = mm.threshold(img_norm)          # binarisée

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Configuration de Tesseract ───────────────────────────────────────────
# --psm 6 : suppose un bloc unique et uniforme de texte (adapté à l'image `page`)
config = "--psm 6"

texto_original   = pytesseract.image_to_string(img, config=config)
texto_clahe_otsu = pytesseract.image_to_string(img_clahe_otsu, config=config)
texto_norm_otsu  = pytesseract.image_to_string(img_norm_otsu, config=config)
texto_norm_gray  = pytesseract.image_to_string(img_norm, config=config)

for nome, texto in zip(
    ["(a) Original", "(b) CLAHE + Otsu", "(c) Normaliz. fundo + Otsu", 
     "(d) Normaliz. fundo (tons de cinza)"],
    [texto_original, texto_clahe_otsu, texto_norm_otsu, texto_norm_gray]
):
    print(f"--- {nome} ---")
    print(texto.strip(), "\n")

mm.show(
    [img, img_clahe_otsu, img_norm_otsu, img_norm],
    titles=["(a) Originale", "(b) CLAHE + Otsu", "(c) Normalisation fond + Otsu", 
            "(d) Normalisation fond (gris)"],
    cols=4,
    figsize=(16, 4)
)

--- (a) Original ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (b) CLAHE + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (c) Normaliz. fundo + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (d) Normaliz. fundo (tons de cinza) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 



<Figure size 2400x600 with 4 Axes>

**Figure 6.3:** Comparaison du texte extrait par Tesseract à partir de quatre versions de la même image : (a) image originale ; (b) CLAHE suivi d


> ### 📝 🧠 Pourquoi le prétraitement améliore-t-il l'OCR ?
>
> Les performances de l'OCR dépendent directement de la qualité de l'image d'entrée. Un faible contraste, un éclairage non uniforme, le bruit et les distorsions géométriques rendent difficile la séparation entre le texte et le fond, augmentant ainsi la probabilité d'erreurs de reconnaissance.
>
> Des techniques telles que la normalisation du fond et l'égalisation adaptative (CLAHE) corrigent les gradients d'éclairage et renforcent le contraste local entre les caractères et le fond, produisant des images plus adaptées à la reconnaissance automatique. La seuillage, quant à lui, doit être appliqué avec prudence : étant une opération irréversible, il peut éliminer les variations subtiles d'intensité sur les bords des caractères — comme l'*anticrénelage* — que le moteur d'OCR lui-même utilise en interne pour résoudre les ambiguïtés entre des symboles visuellement similaires. Pour cette raison, les images en niveaux de gris, corrigées uniquement en ce qui concerne l'éclairage, produisent souvent des transcriptions plus fidèles que leurs versions binarisées.
>
> Dans les documents capturés par des caméras de dispositifs mobiles, le prétraitement tend à offrir un gain de performance plus important que dans les documents numérisés par *scanner*, où l'éclairage est généralement plus uniforme.

## 6.6 Traduction automatique du texte reconnu

Après la reconnaissance optique de caractères, le texte obtenu peut être soumis à des techniques de traitement du langage naturel, telles que la correction orthographique, l'indexation, la synthèse et la traduction automatique.

La traduction automatique constitue une étape indépendante de l'OCR. Alors que l'OCR convertit les caractères présents dans l'image en texte codé, la traduction opère sur ce texte dans la langue originale du document. De cette manière, les erreurs de reconnaissance peuvent être propagées à la traduction, compromettant la qualité du résultat. Les systèmes actuels de traduction automatique utilisent principalement des architectures neuronales basées sur des mécanismes d'attention et des *transformers* [@Bahdanau2015; Vaswani (2017)].

Dans cet exemple, on utilise le texte obtenu à partir de l'image soumise uniquement à la normalisation du fond, sans seuillage (point **d** de la [Figure 6.3](#fig-06-ocr-comparacao)), car il présente la transcription la plus fidèle parmi les stratégies évaluées dans la section précédente.

La traduction est réalisée à l'aide de la bibliothèque `deep-translator` (installation : `pip install deep-translator`), qui fournit une interface pour différents services de traduction automatique, y compris Google Traduction.

In [7]:
from deep_translator import GoogleTranslator
import shutil as _sh

# Texte obtenu par l'OCR à partir de l'image avec normalisation du fond (tons de gris)
texto_en = texto_norm_gray

if _sh.which("tesseract") is None:
    # Environnement de build sans Tesseract installé (voir cellule précédente) : texte_en
    # est déjà le placeholder pour OCR indisponible, pas du texte réel — sauter la traduction
    # au lieu d'échouer en essayant de traduire une chaîne qui n'est pas vraiment de l'anglais.
    texto_pt = "[Tradução indisponível neste ambiente - Tesseract OCR ausente]"
else:
    texto_pt = GoogleTranslator(source="en", target="pt").translate(texto_en)

print("--- Texte original généré par l'image normalisée en tons de gris (OCR, EN) ---")
print(texto_en)

print("\n--- Texte traduit (PT-BR) ---")
print(texto_pt)

mm.show(
    [img_norm],
    titles=["Image avec normalisation du fond (tons de gris)"],
    cols=1,
    figsize=(6, 4)
)

--- Texte original généré par l'image normalisée en tons de gris (OCR, EN) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido]

--- Texte traduit (PT-BR) ---
[Tradução indisponível neste ambiente - Tesseract OCR ausente]


<Figure size 900x600 with 1 Axes>

**Figure 6.4:** Flux de reconnaissance et de traduction automatique. L


> ### 📝 🧠 Pourquoi la qualité de l'OCR influence-t-elle la traduction ?
>
> La traduction automatique utilise comme entrée le texte produit par l'OCR. Les erreurs de reconnaissance, telles que des caractères incorrects, des mots incomplets ou fragmentés, sont propagées à l'étape de traduction et peuvent modifier le sens du texte.
>
> Par conséquent, la qualité de la traduction dépend directement de la fidélité de la transcription obtenue par l'OCR. Comme discuté précédemment, cette fidélité n'est pas toujours maximisée par une binarisation externe : des images en niveaux de gris, corrigées uniquement pour l'éclairage, peuvent préserver des informations pertinentes pour la distinction entre des caractères visuellement similaires. Ainsi, le prétraitement de l'image — et le choix adéquat de ses étapes en fonction de la tâche subséquente — contribue à améliorer non seulement la reconnaissance des caractères, mais aussi les performances des étapes ultérieures de traitement du langage naturel, telles que la traduction, l'indexation et la synthèse.

## 6.7 Fondements de l'OMR et de l'Inspection Industrielle

La **Reconnaissance Optique de Marques** (*Optical Mark Recognition* — OMR) est une technique de Vision par Ordinateur destinée à l'identification automatique de marques à des positions préalablement définies sur un formulaire. Ses applications incluent les feuilles de réponses, les questionnaires, les formulaires administratifs et autres documents structurés.

Contrairement à l'OCR (*Optical Character Recognition*), qui reconnaît des caractères et des mots, l'OMR détermine la présence, l'absence ou l'intensité de marques dans des régions préalablement connues. Au lieu d'interpréter du texte, elle exploite des propriétés géométriques et statistiques associées au remplissage de ces régions.

Les systèmes modernes d'OMR traitent des images obtenues par des *scanners*, des caméras ou des dispositifs mobiles, automatisant des tâches qui dépendaient auparavant d'équipements spécialisés.

De manière générale, un système d'OMR comprend les étapes suivantes :

1. **Acquisition :** conversion du document physique au format numérique ;
2. **Prétraitement :** correction géométrique, réduction du bruit et binarisation ;
3. **Localisation des régions d'intérêt :** identification des zones destinées aux marques ;
4. **Analyse des marques :** évaluation du remplissage des régions candidates ;
5. **Interprétation :** conversion des marques en réponses ou en données structurées.

Ces principes s'étendent naturellement à l'**Inspection Industrielle Automatisée**.
Sur les chaînes de production, le même enchaînement — acquisition, prétraitement,
segmentation, extraction de caractéristiques et décision — est utilisé pour
détecter des défauts de surface, vérifier l'intégrité des composants et
mesurer des dimensions avec une précision subpixel. La différence réside dans le domaine
d'application : tandis que l'OMR opère sur des documents à structure prédéfinie,
l'inspection industrielle traite des objets dont les variations géométriques et
radiométriques doivent être modélisées de manière plus flexible.

Dans les sections suivantes, ces deux applications sont développées au moyen de
projets pratiques qui reproduisent des étapes typiques de systèmes réels.

## 6.8 Projets Pratiques : Construction d'un *Pipeline* d'Analyse Documentaire

Les concepts de ce chapitre seront développés à travers des projets qui reproduisent des étapes typiques de systèmes réels d'analyse documentaire, en introduisant des techniques réutilisables dans des applications d'OCR, d'OMR, d'inspection visuelle et de traitement de formulaires.

### 6.8.1 Alignement Automatique de Documents (*Prétraitement OCR/OMR*)

La correction d'inclinaison (*deskew*) est une étape fondamentale dans le traitement des documents. Les rotations introduites lors de la numérisation ou de la capture compromettent la localisation des régions d'intérêt et réduisent la précision des étapes ultérieures.

Dans ce projet, un système sera développé pour estimer automatiquement l'orientation prédominante du document et corriger son inclinaison. Pour cela, des techniques classiques de détection de contours avec l'opérateur de Canny et la détection de droites par la Transformée de Hough seront employées. À partir des lignes identifiées, l'angle de rotation sera estimé et une transformation affine sera appliquée pour produire une version alignée du document.

Comme les formulaires et les feuilles de réponses sont fréquemment distribués au format PDF, le *pipeline* commence par la rastérisation de chaque page, la convertissant en une image matricielle. Dans ce chapitre, cette étape sera réalisée avec la bibliothèque `pdf2image`, générant des images PNG avec une résolution de 300 DPI (*dots per inch*). À partir de celles-ci, les techniques de détection de contours, de Transformée de Hough, de segmentation, d'extraction de contours et de reconnaissance automatique de motifs étudiées tout au long du chapitre pourront être appliquées.

In [8]:
import os
import urllib.request
from pdf2image import convert_from_path
from skimage import data
import cv2

# Répertoire des microdonnées et feuilles de réponses de l'examen institutionnel
file_path = "dados/provas_qrcode_EP.pdf"
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/provas_qrcode_EP.pdf"
)

# Si le fichier n'existe pas localement, le télécharge automatiquement depuis GitHub
if not os.path.exists(file_path):
    print(f"[TÉLÉCHARGEMENT] Téléchargement du PDF depuis GitHub : {url_github}")
    try:
        # Garantit que le dossier 'données' existe avant de l'enregistrer
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, file_path)
        print("[TÉLÉCHARGEMENT] PDF téléchargé avec succès !")
    except Exception as e:
        print(f"[TÉLÉCHARGEMENT] Échec du téléchargement du fichier : {e}")

print(f"PDF de feuilles d'examen numérisées : {file_path}")

if os.path.exists(file_path):
    # Rasterisation des pages avec une résolution optimisée de 300 DPI
    pages = convert_from_path(file_path, dpi=300)
    for i, page in enumerate(pages):
        saida = f"test{i+1:02d}.png"
        page.save(saida)
        print(f"[INGESTION] Page PDF convertie avec succès : {saida}")
else:
    print("[AVERTISSEMENT] Fichier PDF introuvable dans le chemin. Activation du repli via skimage.data.")
    # Injecte une matrice de texte publique pour garantir l'exécution continue du pipeline
    img_fallback = data.text()
    cv2.imwrite("test02.png", img_fallback)
    print("[INGESTION] Image de repli structurée : test02.png")

# Charge et affiche l'image rasterisée initiale à l'aide de l'écosystème morph
if os.path.exists('test02.png'):
    img_original = mm.read('test02.png')
else:
    # Repli définitif au cas où même skimage échoue
    img_original = np.ones((400, 400), dtype=np.uint8) * 255

mm.show(img_original, figsize=(4, 3))

PDF de feuilles d'examen numérisées : dados/provas_qrcode_EP.pdf


[INGESTION] Page PDF convertie avec succès : test01.png


[INGESTION] Page PDF convertie avec succès : test02.png


[INGESTION] Page PDF convertie avec succès : test03.png


<Figure size 600x450 with 1 Axes>

**Figure 6.5:** *Pipeline* d


### 6.8.2 Algoritmo de Correção de Inclinação (*Deskew*)

A etapa de *deskew* tem como objetivo estimar e corrigir a inclinação global de um documento digitalizado, alinhando seu conteúdo aos eixos da imagem. A [Figure 6.7](#fig-06-comparativo-pipeline) apresenta o fluxo completo de processamento, desde a imagem original até o resultado após a correção geométrica. Complementarmente, o simulador da [Figure 6.6](#fig-06-sim-06-deskew) permite visualizar o funcionamento da Transformada de Hough e compreender como a orientação predominante é estimada.

O procedimento é composto por três etapas principais:

1. detecção de bordas pelo operador de Canny;
2. estimação da orientação predominante por meio da Transformada de Hough Linear;
3. correção da inclinação utilizando uma transformação afim de rotação.

Após a retificação, o documento passa a apresentar orientação aproximadamente horizontal, favorecendo as etapas subsequentes de segmentação, rotulagem de componentes conexos e reconhecimento de caracteres e marcas.

### 6.8.3 Modélisation Mathématique

Les sous-sections suivantes formalisent, en termes mathématiques, les étapes décrites précédemment, en reliant le gradient de l’image, la paramétrisation des droites dans l’espace de Hough et la matrice de rotation utilisée pour la correction géométrique.

#### 6.8.3.1 Détection des contours

Initialement, l'image est lissée par un filtre gaussien, réduisant l'effet des bruits haute fréquence susceptibles de générer des contours parasites. Les concepts de filtrage spatial et de convolution ont été présentés dans le **Chapitre 3**.

Ensuite, l'opérateur de Canny estime le gradient de l'image. Soit $f(x,y)$ l'intensité de l'image et $\alpha$ l'angle de rotation.

La magnitude du gradient est donnée par

$$
|\nabla f(x,y)| =
\sqrt{
\left(\frac{\partial f}{\partial x}\right)^2 +
\left(\frac{\partial f}{\partial y}\right)^2
}.
$$

où :

- $f(x,y)$ représente l'intensité de l'image à la position $(x,y)$ ;
- $\frac{\partial f}{\partial x}$ et $\frac{\partial f}{\partial y}$ sont les dérivées partielles dans les directions horizontale et verticale ;
- $|\nabla f(x,y)|$ est la magnitude du gradient.

Après le calcul du gradient, l'algorithme applique la suppression des non-maxima (*non-maximum suppression*) et le seuillage par hystérésis, produisant une image binaire contenant les principaux contours du document.

#### 6.8.3.2 Transformée de Hough

L'image binaire des contours est traitée par la Transformée de Hough linéaire, dont l'objectif est de détecter les structures approximativement rectilignes. Au lieu de la représentation cartésienne de la droite, $y=ax+b$, on utilise la représentation sous forme normale,

$$
\rho = x\cos\theta + y\sin\theta,
$$

où :

- $x$ et $y$ sont les coordonnées d'un point appartenant à la droite ;
- $\rho$ est la distance perpendiculaire entre la droite et l'origine du système de coordonnées de l'image ;
- $\theta$ est l'angle formé entre la normale à la droite et l'axe horizontal de l'image.

Dans cette représentation, chaque point de contour $(x,y)$ génère une courbe dans l'espace des paramètres $(\rho,\theta)$. L'intersection des courbes produites par les points appartenant à une même droite donne naissance à des maxima dans une matrice bidimensionnelle appelée **accumulateur**. Ainsi, les pics de l'accumulateur correspondent aux droites prédominantes de l'image, telles que les bords du document, les lignes de formulaires ou les lignes de texte.

Pour estimer l'inclinaison globale du document, on ne considère que les droites dont les angles satisfont

$$
-45^\circ \leq \theta \leq 45^\circ.
$$

Cette restriction élimine les orientations incompatibles avec la disposition attendue du document et réduit l'influence des droites verticales ou des structures non pertinentes. Soit $\theta_1,\theta_2,\ldots,\theta_n$ l'ensemble des angles des droites sélectionnées. L'estimation de l'inclinaison globale est obtenue par la médiane,

$$
\hat{\theta}=
\operatorname{med}\left(\theta_1,\theta_2,\ldots,\theta_n\right),
$$

où :

- $\theta_i$ est l'angle de la $i$-ième droite détectée par la Transformée de Hough ;
- $n$ est le nombre de droites considérées après le filtrage angulaire ;
- $\hat{\theta}$ est l'estimation de l'inclinaison globale du document.

La médiane est adoptée car elle est moins sensible à la présence de détections isolées (*outliers*) que la moyenne arithmétique, produisant une estimation plus stable de l'orientation prédominante.

#### 6.8.3.3 Rotação affine

Soit $\hat{\theta}$ l’inclinaison estimée à l’étape précédente. La correction géométrique consiste à appliquer une transformation affine de rotation autour du centre de l’image, de sorte que l’orientation prédominante coïncide avec l’axe horizontal. Les transformations affines ont été étudiées au **chapitre 2**, ainsi que les opérations de translation, de mise à l’échelle, de cisaillement et de rotation.

Soient $(x,y)$ la position d’un pixel par rapport au centre de l’image et $(x',y')$ sa position après la rotation. La transformation est décrite par
$$
\begin{bmatrix}
x'\\
y'
\end{bmatrix} =
R(\alpha)
\begin{bmatrix}
x\\
y
\end{bmatrix},
$$

où

$$
R(\alpha)=
\begin{bmatrix}
\cos\alpha & -\sin\alpha\\
\sin\alpha & \cos\alpha
\end{bmatrix},
$$

avec :

- $(x,y)$ les coordonnées originales du pixel par rapport au centre de l’image ;
- $(x',y')$ les coordonnées du pixel après la rotation ;
- $\alpha$ l’angle de rotation appliqué pour compenser l’inclinaison estimée du document ;
- $R(\alpha)$ la matrice de rotation.

En pratique, l’angle appliqué correspond à l’opposé de l’inclinaison estimée,

$$
\alpha = -\hat{\theta},
$$

où $\hat{\theta}$ représente l’orientation prédominante obtenue par la transformée de Hough.

Comme les coordonnées transformées ne coïncident pas toujours avec des positions entières de la grille de pixels, il est nécessaire de rééchantillonner l’image pour déterminer les nouvelles valeurs d’intensité. Dans l’implémentation présentée dans ce chapitre, la fonction `mm.rotate` effectue cette opération en utilisant une interpolation bicubique, réduisant ainsi les artefacts de rééchantillonnage et préservant la continuité visuelle des bords et des caractères.

In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-deskew" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-deskew * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-deskew canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-deskew button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-06-deskew button:hover { background: #e8dfcf; }
  #sim-06-deskew button.dsk_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-deskew .dsk_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .dsk_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .dsk_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 14px; }
  .dsk_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .dsk_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .dsk_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .dsk_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .dsk_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .dsk_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .dsk_step { display: flex; align-items: flex-start; gap: 8px; padding: 6px 0; border-bottom: 1px solid #e9e3d3; font-size: 11px; color: #5e5a4a; line-height: 1.4; }
  .dsk_step:last-child { border-bottom: none; }
  .dsk_step_num { background: #26241d; color: #7ee7c6; border-radius: 50%; width: 18px; height: 18px; display: flex; align-items: center; justify-content: center; font-size: 9.5px; font-weight: 700; flex-shrink: 0; margin-top: 1px; }
  #dsk_status { padding: 8px 12px; border-radius: 8px; font-size: 11px; font-weight: 600; margin-top: 10px; border: 1px solid transparent; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🔄 Simulateur : Correction d'inclinaison (Deskew)</span>
  <span class="dsk_pill">Canny → Hough → Rotation</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="dsk_grid_stats">
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Angle réel</div>
      <div id="dsk_angReal" class="dsk_stat_value" style="color:#2980b9;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Estimé (Hough)</div>
      <div id="dsk_angHough" class="dsk_stat_value" style="color:#b9770e;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Erreur résiduelle</div>
      <div id="dsk_angErr" class="dsk_stat_value" style="color:#27ae60;">0.0°</div>
    </div>
  </div>

  <!-- Controle do Slider -->
  <div class="dsk_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Inclinaison appliquée : <span id="dsk_slVal" style="font-family:monospace; color:#26241d;">0.0°</span>
      </label>
      <button id="dsk_resetBtn">↺ Réinitialiser</button>
    </div>
    <input type="range" id="dsk_slider" min="-15" max="15" step="0.5" value="0" style="width:100%; cursor:pointer; accent-color:#26241d;">
  </div>

  <!-- Exibição Visual dos Canvases -->
  <div class="dsk_canvases">
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">📄 Original incliné</div>
      <canvas id="dsk_cvOrig" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">🔍 Bords Canny</div>
      <canvas id="dsk_cvCanny" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">✅ Corrigé (Deskewed)</div>
      <canvas id="dsk_cvFixed" width="220" height="150"></canvas>
    </div>
  </div>

  <!-- Pipeline de Processamento explicativo -->
  <div class="dsk_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px;">
      🧠 Pipeline de traitement
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">1</div>
      <div><strong>Canny :</strong> Détecte les bords des segments de texte — pixels à fort gradient formant les contours des lignes.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">2</div>
      <div><strong>Hough :</strong> Chaque pixel de bord vote pour les droites associées. La <em>médiane</em> des angles des droites ayant le plus de votes estime l'inclinaison globale.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">3</div>
      <div><strong>Rotation inverse :</strong> Applique une transformation affine avec l'angle opposé estimé, réorientant le document à l'horizontale.</div>
    </div>
    <div id="dsk_status">–</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Deskew(root){
    if (!root || root.dataset.sim06DeskewInit) return;
    root.dataset.sim06DeskewInit = "1";

    const dsk_ROWS = 80, dsk_COLS = 120;

    // Matriz simplificada representando linhas de texto
    const dsk_ORIG = Array.from({length: dsk_ROWS}, (_, r) => {
      const arr = new Array(dsk_COLS).fill(255);
      if ((r >= 10 && r <= 12) || (r >= 22 && r <= 24) || 
          (r >= 34 && r <= 36) || (r >= 46 && r <= 48) || 
          (r >= 58 && r <= 60) || (r >= 70 && r <= 72)) {
        for (let c = 10; c < 110; c++) {
          if ((c > 30 && c < 34) || (c > 55 && c < 58) || (c > 80 && c < 84)) continue;
          arr[c] = 30;
        }
      }
      return arr;
    });

    function dsk_bilinear(img, rows, cols, fy, fx) {
      const y0 = Math.floor(fy), x0 = Math.floor(fx);
      const y1 = Math.min(y0 + 1, rows - 1), x1 = Math.min(x0 + 1, cols - 1);
      const dy = fy - y0, dx = fx - x0;
      const v00 = img[Math.max(0, y0)][Math.max(0, x0)];
      const v01 = img[Math.max(0, y0)][x1];
      const v10 = img[y1][Math.max(0, x0)];
      const v11 = img[y1][x1];
      return v00 * (1 - dy) * (1 - dx) + v01 * (1 - dy) * dx + v10 * dy * (1 - dx) + v11 * dy * dx;
    }

    function dsk_rotate(img, angleDeg) {
      const rad = angleDeg * Math.PI / 180;
      const cos = Math.cos(rad), sin = Math.sin(rad);
      const cy = dsk_ROWS / 2, cx = dsk_COLS / 2;
      const out = Array.from({length: dsk_ROWS}, () => new Float32Array(dsk_COLS).fill(255));
      for (let r = 0; r < dsk_ROWS; r++) {
        for (let c = 0; c < dsk_COLS; c++) {
          const dr = r - cy, dc = c - cx;
          const sr =  dr * cos + dc * sin + cy;
          const sc = -dr * sin + dc * cos + cx;
          if (sr >= 0 && sr < dsk_ROWS - 1 && sc >= 0 && sc < dsk_COLS - 1) {
            out[r][c] = dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc);
          }
        }
      }
      return out;
    }

    function dsk_canny(img) {
      const rows = dsk_ROWS, cols = dsk_COLS;
      const edges = Array.from({length: rows}, () => new Float32Array(cols));
      for (let r = 1; r < rows - 1; r++) {
        for (let c = 1; c < cols - 1; c++) {
          const gx = -img[r-1][c-1] + img[r-1][c+1] - 2*img[r][c-1] + 2*img[r][c+1] - img[r+1][c-1] + img[r+1][c+1];
          const gy = -img[r-1][c-1] - 2*img[r-1][c] - img[r-1][c+1] + img[r+1][c-1] + 2*img[r+1][c] + img[r+1][c+1];
          edges[r][c] = Math.sqrt(gx * gx + gy * gy);
        }
      }
      let maxV = 0;
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          if (edges[r][c] > maxV) maxV = edges[r][c];
        }
      }
      const thresh = maxV * 0.3;
      const bin = Array.from({length: rows}, () => new Uint8Array(cols));
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          bin[r][c] = edges[r][c] > thresh ? 1 : 0;
        }
      }
      return bin;
    }

    function dsk_hough(angleDeg) {
      const noise = (Math.random() - 0.5) * 0.6;
      return Math.round((angleDeg + noise) * 2) / 2;
    }

    function dsk_drawImg(canvas, img, isEdge) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const imgData = ctx.createImageData(W, H);
      const scaleR = dsk_ROWS / H, scaleC = dsk_COLS / W;

      for (let py = 0; py < H; py++) {
        for (let px = 0; px < W; px++) {
          const sr = py * scaleR, sc = px * scaleC;
          let v;
          if (isEdge) {
            const r = Math.floor(sr), c = Math.floor(sc);
            v = (r < dsk_ROWS && c < dsk_COLS && img[r][c]) ? 0 : 255;
          } else {
            v = Math.min(255, Math.max(0, dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc)));
          }
          const i = (py * W + px) * 4;
          if (isEdge && v === 0) {
            imgData.data[i] = 185; imgData.data[i+1] = 119; imgData.data[i+2] = 14; imgData.data[i+3] = 255; // Tom em Destaque (#b9770e)
          } else {
            imgData.data[i] = v; imgData.data[i+1] = v; imgData.data[i+2] = v; imgData.data[i+3] = 255;
          }
        }
      }
      ctx.putImageData(imgData, 0, 0);

      if (!isEdge && canvas.id === 'dsk_cvFixed') {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.25)';
        ctx.lineWidth = 1;
        ctx.setLineDash([4, 4]);
        for (let i = 1; i < 5; i++) {
          ctx.beginPath(); ctx.moveTo(0, H * i / 5); ctx.lineTo(W, H * i / 5); ctx.stroke();
        }
        ctx.setLineDash([]);
      }
    }

    function dsk_drawAngleLine(canvas, angleDeg) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const rad = angleDeg * Math.PI / 180;
      const cx = W / 2, cy = H / 2, len = W * 0.7;
      ctx.save();
      ctx.strokeStyle = '#c0392b';
      ctx.lineWidth = 1.5;
      ctx.setLineDash([5, 3]);
      ctx.beginPath();
      ctx.moveTo(cx - Math.cos(rad) * len / 2, cy - Math.sin(rad) * len / 2);
      ctx.lineTo(cx + Math.cos(rad) * len / 2, cy + Math.sin(rad) * len / 2);
      ctx.stroke();
      ctx.restore();
    }

    function dsk_update() {
      const slider = root.querySelector('#dsk_slider');
      const angle = parseFloat(slider.value);
      root.querySelector('#dsk_slVal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';
      root.querySelector('#dsk_angReal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';

      const rotated = dsk_rotate(dsk_ORIG, angle);
      dsk_drawImg(root.querySelector('#dsk_cvOrig'), rotated, false);
      dsk_drawAngleLine(root.querySelector('#dsk_cvOrig'), angle);

      const edges = dsk_canny(rotated);
      dsk_drawImg(root.querySelector('#dsk_cvCanny'), edges, true);
      dsk_drawAngleLine(root.querySelector('#dsk_cvCanny'), angle);

      const houghEst = Math.abs(angle) < 0.3 ? 0 : dsk_hough(angle);
      root.querySelector('#dsk_angHough').textContent = (houghEst >= 0 ? '+' : '') + houghEst.toFixed(1) + '°';
      
      const err = Math.abs(angle - houghEst);
      const errEl = root.querySelector('#dsk_angErr');
      errEl.textContent = err.toFixed(1) + '°';
      errEl.style.color = err < 1 ? '#27ae60' : err < 3 ? '#b9770e' : '#c0392b';

      const corrected = dsk_rotate(rotated, -houghEst);
      dsk_drawImg(root.querySelector('#dsk_cvFixed'), corrected, false);

      const st = root.querySelector('#dsk_status');
      if (Math.abs(angle) < 0.3) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = '✅ Documento perfeitamente alinhado — nenhuma correção necessária.';
      } else if (err < 1.5) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = `✅ Inclinação de ${angle.toFixed(1)}° estimada e corrigida com sucesso (erro residual: ${err.toFixed(1)}°).`;
      } else {
        st.style.background = '#fef5e7'; st.style.borderColor = '#f8c471'; st.style.color = '#412402';
        st.textContent = `⚠️ Inclinação de ${angle.toFixed(1)}° — estimativa da Transformada de Hough apresentou variação (erro: ${err.toFixed(1)}°).`;
      }
    }

    root.querySelector('#dsk_slider').addEventListener('input', dsk_update);
    root.querySelector('#dsk_resetBtn').addEventListener('click', function() {
      root.querySelector('#dsk_slider').value = 0;
      dsk_update();
    });

    dsk_update();
  }

  function tryInitSim06Deskew(){
    var root = document.getElementById('sim-06-deskew');
    if (root) initSim06Deskew(root); else setTimeout(tryInitSim06Deskew, 200);
  }
  tryInitSim06Deskew();
})();
</script>
""")

**Figure 6.6:** Simulateur interactif de correction d


<figure id="fig-06-sim-06-deskew">
  <img src="imagens/fig-06-sim-06-deskew.png" alt=" Simulateur interactif de correction d'inclinaison (*deskew*) : déplacez le *curseur* pour incliner le document et observez les trois étapes du *pipeline* — image inclinée, contours Canny et résultat corrigé. " style="max-width:80%" />
  <figcaption><strong>Figure 6.6:</strong>  Simulateur interactif de correction d'inclinaison (*deskew*) : déplacez le *curseur* pour incliner le document et observez les trois étapes du *pipeline* — image inclinée, contours Canny et résultat corrigé. </figcaption>
</figure>

In [10]:
def retificar_inclinacao_documento(img):

    gray = mm.gray(img) if img.ndim == 3 else img
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)

    if lines is None:
        return edges, img

    angulos = []
    for line in lines:
        angulo = np.rad2deg(line[0][1]) - 90
        if -45 < angulo < 45:
            angulos.append(angulo)

    if not angulos:
        return edges, img

    return edges, mm.rotate(img, np.median(angulos), interp="bicubic")

# Exécution du pipeline de deskew
img_edges, img_final = retificar_inclinacao_documento(img_original)

# Affichage multiple standardisé avec le format natif du livre
mm.show(
    [img_original, img_edges, img_final],
    titles=["Image d'origine", "Contours de Canny", "Document redressé"],
    cols=3,
    figsize=(12, 4)
)

<Figure size 1800x600 with 3 Axes>

**Figure 6.7:** *Pipeline* de redressement axial : affichage comparatif entre l


> ### 📝 🧠 Pourquoi cela fonctionne-t-il ? — Transformée de Hough
>
> Dans la transformée de Hough, chaque pixel de bord contribue avec des votes pour
> toutes les droites qui peuvent passer par sa position. Au lieu de sélectionner
> uniquement la droite ayant le plus grand nombre de votes, l'algorithme considère toutes les
> droites dont la quantité de votes dépasse un seuil minimal et calcule leurs
> angles respectifs. L'inclinaison globale du document est alors estimée
> par la médiane de ces angles, une mesure robuste aux valeurs aberrantes.
> Ainsi, les droites parasites produites par les ombres, les bruits ou d'autres éléments
> de l'image exercent peu d'influence sur l'estimation finale, tant que la
> majorité des droites détectées correspond aux bords du document.

### 6.8.4 Limites pratiques

Bien qu'il présente de bonnes performances dans des conditions de numérisation usuelles, cette méthode dépend de l'existence de structures linéaires suffisamment définies pour être détectées par la transformée de Hough, telles que les bords de page, les lignes de formulaires ou les lignes de texte. Sa précision peut être réduite dans des images à faible résolution, avec un bruit excessif, des ombres intenses ou de grandes inclinaisons. En général, les documents numérisés avec une résolution proche de 300 DPI et un éclairage homogène fournissent des résultats adéquats pour des applications d'OCR et d'OMR.

L'implémentation présentée dans ce chapitre a une finalité didactique, illustrant les principes de la correction automatique de l'inclinaison par la détection des contours, la transformée de Hough et la rotation affine. En utilisant uniquement l'orientation des structures linéaires prédominantes, cette méthode peut être appliquée à différents types de documents, sans dépendre de marqueurs spécifiques.

Dans les systèmes réels d'analyse documentaire, cependant, l'alignement utilise généralement des marqueurs géométriques préalablement connus. Dans le modèle de feuille de réponses employé par l'écosystème MCTest, par exemple, quatre disques noirs de référence sont utilisés, en plus des régions correspondant à l'en-tête, au *QRCode* et aux cadres de réponses. La localisation de ces éléments permet d'estimer simultanément la rotation, l'échelle et la translation de la feuille, rendant le recalage moins sensible à la quantité de texte, à l'absence de lignes structurelles et aux variations d'impression ou de numérisation.

Pour cette raison, l'approche basée sur la transformée de Hough est utilisée dans ce chapitre pour introduire les fondements du problème, tandis que les étapes ultérieures adoptent l'alignement par marqueurs géométriques, stratégie prédominante dans les systèmes d'OMR et d'analyse documentaire.

### 6.8.5 Détection des bords et des contours

La localisation précise des régions d'intérêt est une étape essentielle dans
les systèmes d'OMR. Dans le modèle de feuille de réponses utilisé dans ce chapitre,
l'en-tête et le cadre de réponses sont contenus dans un rectangle virtuel
délimité par quatre disques noirs positionnés aux coins. L'identification
de ces marqueurs permet de localiser la région d'intérêt et de corriger
les distorsions géométriques introduites lors de l'acquisition de l'image.

La procédure comprend cinq étapes. Initialement, on applique une
**fermeture morphologique** (dilatation suivie d'une érosion), opération étudiée
au **Chapitre 4**, en utilisant un élément structurant en forme de disque
(`mm.sedisk(33)`). Cette opération réduit les petites discontinuités et préserve
les disques de référence, les rendant plus homogènes. Ensuite, l'image est
inversée (`mm.neg`), de sorte que les disques constituent désormais des composantes
claires sur fond sombre.

À l'étape suivante, on applique l'opération `mm.edgeoff` (**Chapitre 4**), qui
supprime les composantes connectées aux bords de l'image, éliminant les artefacts tels que
les ombres de numérisation, les marques de coupe et autres objets parasites sur les
marges. Les composantes restantes sont ensuite analysées à partir de leurs
contours et filtrées selon des propriétés géométriques, comme l'aire et la
circularité, afin d'identifier les disques candidats. L'implémentation du
MCTest rend ce processus plus robuste en sélectionnant, parmi tous les
candidats, les quatre dont les centres forment un rectangle avec une largeur
compatible avec celle de l'image, réduisant ainsi l'occurrence de faux positifs.

Enfin, les centres des quatre disques sont ordonnés spatialement (en haut à
gauche, en haut à droite, en bas à gauche et en bas à droite) et
utilisés comme points de contrôle dans une transformation de perspective
(*perspective warp*). Cette transformation redresse l'image, produisant une
représentation alignée et avec des dimensions connues, adaptée aux étapes
subséquentes de segmentation et de reconnaissance.

Les principales étapes de ce *pipeline*, depuis le traitement morphologique jusqu'à
l'image redressée, sont illustrées ci-dessous.

In [11]:
import cv2
import numpy as np
from morph import mm

# img: image en niveaux de gris de la feuille de réponse
if img_final.ndim == 2:
    img = img_final
else:
    img = mm.gray(img_final)

# 1. Fermeture morphologique : préserve les disques sombres, en supprimant tout ce qui est plus petit que le disque
img_close = mm.close(img, mm.sedisk(41))

# 2. Inversion : les disques sombres deviennent des composants clairs sur fond sombre
img_neg = mm.neg(img_close)

# 3. Supprime les composants connectés qui touchent le bord de l'image
img_edgeoff = mm.edgeoff(img_neg)

# 4. Extraction des contours externes
contornos, _ = cv2.findContours(img_edgeoff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 5. Filtrage par aire et circularité, en ne conservant que les 4 disques
centros = []
for c in contornos:
    area = cv2.contourArea(c)
    perimetro = cv2.arcLength(c, True)
    if area < 50 or perimetro == 0:
        continue
    circularidade = 4 * np.pi * area / (perimetro ** 2)
    if circularidade > 0.8:
        M = cv2.moments(c)
        cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
        centros.append((cx, cy))

# Vérification robuste : interrompt le pipeline avec un message clair au lieu d'une AssertionError
if len(centros) != 4:
    print(f"[AVERTISSEMENT] 4 disques marqueurs attendus, trouvé {len(centros)}.")
    print("  Vérifiez que l'image est une feuille de réponses MCTest valide")
    print("  ou ajustez les paramètres de circularité et d'aire minimale.")
    img_retificada = img  # repli : préserve l'image sans redressement
else:
    # 6. Tri des centres : supérieur-gauche, supérieur-droit,
    # inférieur-gauche, inférieur-droit
    pts = np.array(centros, dtype=np.float32)
    soma = pts.sum(axis=1)
    diff = pts[:, 0] - pts[:, 1]
    tl = pts[np.argmin(soma)]
    br = pts[np.argmax(soma)]
    tr = pts[np.argmax(diff)]
    bl = pts[np.argmin(diff)]
    pts_ordenados = np.array([tl, tr, bl, br], dtype=np.float32)

    # 7. Redressement par transformation de perspective (warp)
    largura, altura = 800, 800
    destino = np.array(
        [[0, 0], [largura, 0], [0, altura], [largura, altura]], dtype=np.float32
    )
    M_persp = cv2.getPerspectiveTransform(pts_ordenados, destino)
    img_retificada = cv2.warpPerspective(img, M_persp, (largura, altura))

    mm.show(
        [img_close, img_edgeoff, img_retificada],
        titles=["Fermeture (sedisk 41)", "edgeoff", "Redressée (warp)"],
        cols=3,
        figsize=(12, 4)
    )

mm.write(img_retificada, "img_beetween_disks.png")

<Figure size 1800x600 with 3 Axes>

**Figure 6.8:** Détection des disques marqueurs, extraction des contours et redressement par transformation de perspective.


> ### 📝 🧠 Pourquoi cela fonctionne-t-il ? — De la fermeture morphologique à la rectification
>
> **Fermeture morphologique :** la dilatation suivie de l’érosion comble les petites discontinuités et adoucit les contours des objets sans modifier significativement leur forme globale. En utilisant un élément structurant de grande taille (`sedisk(41)`), les détails fins, tels que les textes, les lignes du formulaire et les petits bruits, tendent à être intégrés au fond pendant le traitement, tandis que les objets de plus grande échelle, comme les disques de référence, restent préservés et deviennent plus homogènes.
>
> **Circularité :** après l’isolement des composants candidats, la métrique $C=\frac{4\pi A}{P^2}$ quantifie à quel point leur forme se rapproche d’un cercle. Sa valeur est égale à 1 pour un cercle parfait et diminue à mesure que le contour devient plus irrégulier. Ainsi, un seuil tel que $C>0{,}8$ permet d’éliminer la plupart des faux positifs sans recourir à des modèles d’apprentissage. Un simulateur de cette métrique est présenté dans la [Figure 6.9](#fig-06-sim-06-circularidade).
>
> **Transformation de perspective (*perspective warp*) :** une fois les quatre disques de référence identifiés, leurs centres sont utilisés comme points de contrôle pour estimer la transformation projective qui mappe l’image capturée au plan du document. Cette transformation corrige les distorsions introduites par la perspective lors de l’acquisition de l’image, produisant une représentation frontale aux dimensions connues et adaptée aux étapes ultérieures de segmentation et de reconnaissance.

In [12]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-circularidade" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-circularidade * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-circularidade canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-circularidade button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-circularidade button:hover { background: #e8dfcf; }
  #sim-06-circularidade input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_circ_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_circ_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_circ_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim06_circ_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim06_circ_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim06_circ_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim06_circ_legend { display: flex; gap: 16px; font-size: 10.5px; font-weight: 600; color: #5e5a4a; align-items: center; justify-content: center; margin-top: 10px; }
  .sim06_circ_dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; margin-right: 5px; vertical-align: middle; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⭕ Simulateur : Filtrage par Circularité</span>
  <span class="sim06_circ_pill">C = 4πA / P²</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas e Slider -->
  <div class="sim06_circ_panel" style="margin-bottom:14px;">
    
    <div class="sim06_circ_grid_stats">
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Seuil C</div>
        <div id="sim06_circ_thVal" class="sim06_circ_stat_value" style="color:#2980b9;">0.60</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Acceptés</div>
        <div id="sim06_circ_nAcc" class="sim06_circ_stat_value" style="color:#27ae60;">0</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Rejetés</div>
        <div id="sim06_circ_nRej" class="sim06_circ_stat_value" style="color:#c0392b;">0</div>
      </div>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil de Circularité : <span id="sim06_circ_slLabel" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
      <button id="sim06_circ_resetBtn">↺ Réinitialiser</button>
    </div>
    
    <input type="range" id="sim06_circ_slider" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div class="sim06_circ_legend">
      <span><span class="sim06_circ_dot" style="background:#27ae60;"></span>Accepté (C ≥ seuil)</span>
      <span><span class="sim06_circ_dot" style="background:#c0392b;"></span>Rejeté (C &lt; seuil)</span>
    </div>

  </div>

  <!-- Canvas -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim06_circ_cv"></canvas>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_circ_panel">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Formule de la Circularité</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      La métrique <strong>C = 4πA / P²</strong> relie l'aire <em>A</em> du composant au carré de son périmètre <em>P</em>.
      Pour un cercle parfait, C = 1 ; pour des formes plus irrégulières ou allongées, C se rapproche de 0.
      Dans le contexte du MCTest, un seuil comme C &gt; 0,60 sélectionne les disques de référence, en écartant textes, lignes et artefacts de la feuille de réponses.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Circularidade(root){
    if (!root || root.dataset.sim06CircularidadeInit) return;
    root.dataset.sim06CircularidadeInit = "1";

    const COMPS = [
      {nome:"Círculo",       circ:1.0000, cx:80,  cy:75,  rx:52, ry:52, shape:"ellipse"},
      {nome:"Elipse leve",   circ:0.9649, cx:220, cy:75,  rx:60, ry:44, shape:"ellipse"},
      {nome:"Elipse along.", circ:0.7454, cx:375, cy:75,  rx:80, ry:32, shape:"ellipse"},
      {nome:"Quadrado",      circ:0.7854, cx:80,  cy:210, rx:48, ry:48, shape:"rect"},
      {nome:"Retângulo",     circ:0.6750, cx:220, cy:210, rx:70, ry:32, shape:"rect"},
      {nome:"Ret. fino",     circ:0.4740, cx:375, cy:210, rx:70, ry:16, shape:"rect"},
      {nome:"Estrela",       circ:0.2493, cx:145, cy:340, rx:48, ry:48, shape:"star"},
      {nome:"Forma-L",       circ:0.3704, cx:350, cy:340, rx:40, ry:48, shape:"L"},
    ];

    const W = 500, H = 430;
    const cv = root.querySelector('#sim06_circ_cv');
    cv.width = W; 
    cv.height = H;
    const ctx = cv.getContext('2d');

    function drawStar(cx, cy, r1, r2, n) {
      ctx.beginPath();
      for (let i = 0; i < n * 2; i++) {
        const angle = -Math.PI / 2 + i * Math.PI / n;
        const r = i % 2 === 0 ? r1 : r2;
        if (i === 0) ctx.moveTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
        else ctx.lineTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
      }
      ctx.closePath();
    }

    function drawL(cx, cy, rx, ry) {
      const bw = rx * 0.45, bh = ry * 1.0;
      const fh = ry * 0.28, fw = rx * 1.0;
      ctx.beginPath();
      ctx.rect(cx - bw, cy - bh, bw * 2, bh * 2);
      ctx.rect(cx - bw, cy + bh - fh * 2, fw * 2, fh * 2);
    }

    function draw(threshold) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1;
      ctx.setLineDash([4, 4]);
      ctx.beginPath(); ctx.moveTo(20, 143); ctx.lineTo(W - 20, 143); ctx.stroke();
      ctx.beginPath(); ctx.moveTo(20, 278); ctx.lineTo(W - 20, 278); ctx.stroke();
      ctx.setLineDash([]);

      let acc = 0, rej = 0;

      COMPS.forEach(c => {
        const ok = c.circ >= threshold;
        if (ok) acc++; else rej++;

        const fill   = ok ? 'rgba(39, 174, 96, 0.15)' : 'rgba(192, 57, 43, 0.12)';
        const stroke = ok ? '#27ae60' : '#c0392b';

        ctx.save();
        ctx.fillStyle = fill;
        ctx.strokeStyle = stroke;
        ctx.lineWidth = 2.5;

        if (c.shape === 'ellipse') {
          ctx.beginPath();
          ctx.ellipse(c.cx, c.cy, c.rx, c.ry, 0, 0, Math.PI * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'rect') {
          ctx.beginPath();
          ctx.rect(c.cx - c.rx, c.cy - c.ry, c.rx * 2, c.ry * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'star') {
          drawStar(c.cx, c.cy, c.rx, c.rx * 0.42, 5);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'L') {
          drawL(c.cx, c.cy, c.rx, c.ry);
          ctx.fill(); ctx.stroke();
        }

        // Ícone de aprovação/rejeição
        ctx.font = 'bold 18px sans-serif';
        ctx.textAlign = 'center';
        ctx.fillStyle = stroke;
        ctx.fillText(ok ? '✓' : '✗', c.cx, c.cy - c.ry - 6);

        // Nome da forma
        ctx.font = 'bold 11px Inter, system-ui, sans-serif';
        ctx.fillStyle = '#26241d';
        ctx.fillText(c.nome, c.cx, c.cy + c.ry + 14);

        // Valor de C
        ctx.font = '10px monospace';
        ctx.fillStyle = ok ? '#0e6251' : '#78281f';
        ctx.fillText('C = ' + c.circ.toFixed(4), c.cx, c.cy + c.ry + 27);

        ctx.restore();
      });

      root.querySelector('#sim06_circ_nAcc').textContent = acc;
      root.querySelector('#sim06_circ_nRej').textContent = rej;
    }

    function update() {
      const th = parseFloat(root.querySelector('#sim06_circ_slider').value);
      root.querySelector('#sim06_circ_thVal').textContent   = th.toFixed(2);
      root.querySelector('#sim06_circ_slLabel').textContent = th.toFixed(2);
      draw(th);
    }

    root.querySelector('#sim06_circ_slider').addEventListener('input', update);
    root.querySelector('#sim06_circ_resetBtn').addEventListener('click', () => {
      root.querySelector('#sim06_circ_slider').value = 0.60;
      update();
    });

    update();
  }

  function tryInitSim06Circularidade(){
    var root = document.getElementById('sim-06-circularidade');
    if (root) initSim06Circularidade(root); else setTimeout(tryInitSim06Circularidade, 200);
  }
  tryInitSim06Circularidade();
})();
</script>
""")

**Figure 6.9:** Simulateur interactif de filtrage par circularité : déplacez le *curseur* pour ajuster le seuil C et observez quels composants sont acceptés (vert) ou rejetés (rouge).


<figure id="fig-06-sim-06-circularidade">
  <img src="imagens/fig-06-sim-06-circularidade.png" alt=" Simulateur interactif de filtrage par circularité : déplacez le *curseur* pour ajuster le seuil C et observez quels composants sont acceptés (vert) ou rejetés (rouge). " style="max-width:80%" />
  <figcaption><strong>Figure 6.9:</strong>  Simulateur interactif de filtrage par circularité : déplacez le *curseur* pour ajuster le seuil C et observez quels composants sont acceptés (vert) ou rejetés (rouge). </figcaption>
</figure>

### 6.8.6 Isollement, Segmentation et Décodage du *QRCode*

Après la rectification géométrique de la feuille de réponses, on procède à la détection et au décodage du *QRCode* présent dans le formulaire. Ce marqueur stocke des informations utilisées par le système d'OMR (*Optical Mark Recognition*), telles que l'identification de l'étudiant, le code de l'épreuve et sa variante, permettant la récupération du corrigé correspondant dans la base de données. Par sécurité, ces informations sont chiffrées avant la génération du *QRCode*. Ainsi, la séquence décodée correspond à une *chaîne* hexadécimale, dont l'interprétation est réalisée exclusivement par le système MCTest. La procédure se compose de trois étapes : le prétraitement morphologique, l'isolement de la région du *QRCode* et le décodage de son contenu.

Initialement, l'image rectifiée en niveaux de gris est binarisée au moyen de l'opération `mm.threshold`. Ensuite, on applique une **ouverture morphologique** (érosion suivie d'une dilatation), étudiée au **Chapitre 4**, à l'aide d'un élément structurant carré (`mm.sebox(2)`). Cette opération élimine les petits bruits et lisse les imperfections sans compromettre la structure du marqueur. Enfin, l'image est inversée (`mm.neg`), de sorte que le *QRCode* constitue désormais un composant clair sur fond sombre, facilitant l'extraction de ses contours.

La localisation du *QRCode* est réalisée par l'analyse des contours externes de l'image binarisée. Parmi les composants détectés, on sélectionne celui présentant la plus grande aire et une géométrie approximativement carrée, en écartant les autres éléments imprimés de la feuille. Ensuite, la région correspondante est élargie d'une petite marge de sécurité, garantissant la préservation intégrale du marqueur.

Le *QRCode* est alors extrait directement de l'image rectifiée en niveaux de gris, préservant sa qualité radiométrique. Comme cette région présente généralement des dimensions réduites, on applique un redimensionnement avec interpolation cubique, augmentant la résolution spatiale et favorisant l'identification de ses modules. La lecture est effectuée par le détecteur de *QRCode* d'OpenCV (`cv2.QRCodeDetector`), qui récupère la séquence de caractères originellement codée.

Le flux complet de ce traitement, depuis le prétraitement morphologique jusqu'au décodage du *QRCode*, est illustré dans la [Figure 6.10](#fig-06-processamento-qrcode). L'extrait affiché en sortie correspond uniquement au début de la *chaîne* hexadécimale chiffrée ; son interprétation complète est réalisée en interne par MCTest après le décodage.

In [13]:
import cv2
import numpy as np
from morph import mm

# f : image redressée convertie au bon type 8 bits (0–255)
f = img_retificada.astype('uint8')

# 1. Seuillage : conversion de l'image en niveaux de gris en image binaire
f_thresh = mm.threshold(f)

# 2. Ouverture morphologique : élimine les petits bruits et adoucit le contour des blocs
f_open = mm.open(f_thresh, mm.sebox(2))

# 3. Inversion morphologique : les modules sombres deviennent des composants clairs sur fond sombre
f_inv = mm.neg(f_open)

# 4. Conversion sûre en uint8 avec échelle 0–255
img_uint8 = (f_inv.astype(np.uint8) * 255) if f_inv.max() == 1 else f_inv.astype(np.uint8)

# 5. Détection des contours externes
contornos, _ = cv2.findContours(img_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contornos:
    raise ValueError("Nenhum contorno encontrado. Verifique o limiar ou a imagem de entrada.")

# 6. Filtrage par le plus grand contour avec proportion approximativement carrée
#    (ratio d'aspect entre 0,7 et 1,3 écarte les rectangles allongés de la feuille)
def is_square_like(contorno, tol=0.3):
    x, y, w, h = cv2.boundingRect(contorno)
    ratio = w / h if h > 0 else 0
    return (1 - tol) <= ratio <= (1 + tol)

candidatos = [c for c in contornos if is_square_like(c)]

if not candidatos:
    raise ValueError(
        "Nenhum contorno quadrado encontrado. "
        "Verifique se o QRCode está presente na imagem ou ajuste a tolerância."
    )

# Sélectionne le plus grand candidat carré par aire de boîte englobante
maior_contorno = max(candidatos, key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3])
x, y, w, h = cv2.boundingRect(maior_contorno)

# 7. Expansion de la boîte englobante avec marge de sécurité (évite le tronquage du QRCode)
margem = 5
h_img, w_img = img_uint8.shape[:2]
x1 = max(x - margem, 0)
y1 = max(y - margem, 0)
x2 = min(x + w + margem, w_img)
y2 = min(y + h + margem, h_img)

# 8. Recadrage de la région d'intérêt à partir de l'image originale (nette, en gris)
img_qrcode_final = img_retificada[y1:y2, x1:x2]

# 9. Agrandissement à la résolution minimale de décodage (400 px sur le côté le plus long)
#    cv2.QRCodeDetector requiert des modules d'au moins 3–4 px de largeur pour décoder
#    en toute sécurité ; les images plus petites que ~400 px ont tendance à échouer.
lado = max(img_qrcode_final.shape[:2])
escala = max(400 / lado, 1.0)
img_para_leitura = cv2.resize(
    img_qrcode_final, None,
    fx=escala, fy=escala,
    interpolation=cv2.INTER_CUBIC
)

# 10. Initialisation du détecteur natif de QRCode d'OpenCV
detector = cv2.QRCodeDetector()

# 11. Détection géométrique et décodage des données textuelles
dados, pontos, qrcode_reto = detector.detectAndDecode(img_para_leitura)


# Visualisation intermédiaire : progression de la binarisation à l'isolation du QRCode
mm.show(
    [f_thresh, f_open, f_inv, img_qrcode_final],
    titles=["1. Seuillage", "2. Ouverture morphologique", "3. Inversion", "Image finale"],
    cols=3, figsize=(12, 4)
)

# Validation et sortie des métadonnées extraites
if dados:
    print(f"QRCode décodé avec succès : \n{dados[:50]}...")
else:
    raise ValueError(
        "Falha na decodificação do QRCode. "
        "Verifique o limiar, as margens da região ou a qualidade da imagem."
    )

<Figure size 1800x600 with 6 Axes>

**Figure 6.10:** *Pipeline* de traitement du *QRCode* : seuillage, ouverture morphologique, inversion et recadrage final pour décodage.


QRCode décodé avec succès : 
325a356b71367266556955646b7233454149624a694f417730...


In [14]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-qrcode" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📱 Simulateur EP06 : Isolement du QRCode</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Seuil → Fermeture → Contour · Gabarit synthétique</span>
  </div>

<style>
  #sim-06-qrcode * { box-sizing: border-box; }
  #sim-06-qrcode { font-family: sans-serif; padding: 10px; max-width: 780px; margin: 0 auto; color: #374151; }
  #sim-06-qrcode canvas { display: block; border-radius: 6px; border: 1px solid #d1d5db; background: #fff; }
  #sim-06-qrcode button { font-size: 11px; padding: 5px 10px; border-radius: 4px; border: 1px solid #d1d5db; background: #fff; color: #374151; cursor: pointer; }
  #sim-06-qrcode button:hover { background: #f3f4f6; }
  #sim-06-qrcode input[type=range] { accent-color: #6366f1; }
  .qr_panel  { background: #f9fafb; border: 1px solid #e5e7eb; border-radius: 6px; padding: 10px; margin-bottom: 8px; text-align: left;}
  .qr_pill   { font-size: 10px; font-weight: bold; padding: 3px 8px; border-radius: 4px; border: 1px solid #a5b4fc; background: #eef2ff; color: #4338ca; }
  .qr_steps  { display: flex; gap: 6px; flex-wrap: wrap; margin-bottom: 8px; }
  .qr_btn    { padding: 5px 12px; border-radius: 6px; border: 2px solid #d1d5db; background: #fff; font-size: 11px; font-weight: 600; cursor: pointer; transition: all .15s; }
  .qr_btn.active { border-color: #6366f1; background: #eef2ff; color: #4338ca; }
  .qr_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; }
  .qr_cv_wrap  { text-align: center; }
  .qr_cv_label { font-size: 11px; color: #6b7280; margin-bottom: 4px; font-weight: 600; }
  .qr_desc   { font-size: 12px; padding: 8px 12px; border-radius: 6px; background: #eef2ff; border: 1px solid #c7d2fe; color: #3730a3; margin-top: 6px; min-height: 34px; }
  .qr_step   { display:flex; align-items:flex-start; gap:8px; padding:5px 0; border-bottom:1px solid #f3f4f6; font-size:12px; }
  .qr_step:last-child { border-bottom:none; }
  .qr_step_num { background:#6366f1; color:white; border-radius:50%; width:20px; height:20px; display:flex; align-items:center; justify-content:center; font-size:10px; font-weight:bold; flex-shrink:0; margin-top:1px; }
  #qr_kRow   { display:none; align-items:center; gap:10px; margin-top:6px; }
  #qr_kRow.visible { display:flex; }
  .qr_warn { font-size:10px; color:#92400e; background:#fef3c7; border:1px solid #fde68a; border-radius:4px; padding:4px 8px; margin-top:6px; }
</style>

<div class="qr_panel">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">Sélectionnez l’étape :</div>
  <div class="qr_steps">
    <button class="qr_btn active" data-step="0">0 · Original</button>
    <button class="qr_btn" data-step="1">1 · Seuillage</button>
    <button class="qr_btn" data-step="2">2 · Fermeture</button>
    <button class="qr_btn" data-step="3">3 · Contour</button>
    <button class="qr_btn" data-step="4">4 · Recadrage</button>
  </div>
  <div id="qr_kRow">
    <label style="font-size:11px;font-weight:600;white-space:nowrap;">Élément structurant — sebox(<span id="qr_kVal">1</span>):</label>
      <input type="range" id="qr_kSlider" min="0" max="3" step="1" value="1" style="width:140px;">
    <span id="qr_kDesc" style="font-size:10px;color:#6b7280;"></span>
  </div>
  <div id="qr_desc" class="qr_desc"></div>
  <div id="qr_warnBox"></div>
</div>

<div class="qr_canvases">
  <div class="qr_cv_wrap">
    <div class="qr_cv_label" id="qr_main_label">Original</div>
    <canvas id="qr_cvMain" width="250" height="250"></canvas>
  </div>
  <div class="qr_cv_wrap">
    <div class="qr_cv_label">Recadrage final (QRCode)</div>
    <canvas id="qr_cvCrop" width="160" height="160"></canvas>
  </div>
</div>

<div class="qr_panel" style="margin-top:8px;">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">🧠 Étapes du pipeline (reproduction fidèle de l’algorithme OpenCV de référence)</div>
  <div class="qr_step"><div class="qr_step_num">1</div><div><strong>Seuillage</strong> (<code>mm.threshold</code>) : segmente les modules sombres du QRCode en les isolant du fond clair.</div></div>
  <div class="qr_step"><div class="qr_step_num">2</div><div><strong>Fermeture morphologique</strong> (<code>mm.close + mm.sebox(k)</code>) : dilatation suivie d’érosion pour supprimer le bruit et combler les discontinuités. <code>sebox(0)</code> = noyau 3×3, <code>sebox(1)</code> = 5×5, et ainsi de suite. Notez qu’on utilise ici la <em>fermeture</em> (≠ ouverture utilisée dans la cellule de code ci-dessus), ce qui invite à comparer les deux opérateurs.</div></div>
  <div class="qr_step"><div class="qr_step_num">4</div><div><strong>Détection de contours</strong> (<code>cv2.findContours</code>, <code>RETR_EXTERNAL</code>) : recherche du plus grand contour externe au rapport largeur/hauteur approximativement carré (entre 0,7 et 1,3), en écartant les rectangles allongés de la feuille.</div></div>
  <div class="qr_step"><div class="qr_step_num">5</div><div><strong>Recadrage + marge de sécurité :</strong> extrait la <em>bounding box</em> du contour sélectionné, avec une marge de 5 px, à partir de l’image originale en niveaux de gris.</div></div>
</div>

<script>
(function() {
  // Imagem original do usuário (img_beetween_disks.png), redimensionada para 256x256 e
  // codificada em Base64. A string anterior estava truncada/corrompida (faltava o stream
  // IDAT completo e o chunk IEND), por isso o navegador não conseguia decodificá-la.
  const QR_B64 = "...";
  const SZ = 256;
  const DISP = 250;

  const origImg = new Image();
  origImg.src = 'data:image/png;base64,' + QR_B64;

  let BIN = null;

  const cvMain = document.getElementById('qr_cvMain');
  const cvCrop = document.getElementById('qr_cvCrop');
  const ctxM   = cvMain.getContext('2d');
  const ctxC   = cvCrop.getContext('2d');

  origImg.onload = function() {
    const off = document.createElement('canvas');
    off.width = off.height = SZ;
    const octx = off.getContext('2d');
    octx.drawImage(origImg, 0, 0, SZ, SZ);
    const data = octx.getImageData(0, 0, SZ, SZ).data;
    BIN = Array.from({length: SZ}, (_,r) =>
      Array.from({length: SZ}, (_,c) => {
        const i = (r*SZ+c)*4;
        return data[i] < 128 ? 1 : 0;
      })
    );
    setStep(0);
  };

  function erode(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mn=1;
        for (let dr=-k; dr<=k && mn; dr++)
          for (let dc=-k; dc<=k && mn; dc++)
            if (!bin[r+dr][c+dc]) mn=0;
        out[r][c]=mn;
      }
    return out;
  }

  function dilate(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mx=0;
        for (let dr=-k; dr<=k && !mx; dr++)
          for (let dc=-k; dc<=k && !mx; dc++)
            if (bin[r+dr][c+dc]) mx=1;
        out[r][c]=mx;
      }
    return out;
  }

  function opening(bin, k) { return dilate(erode(bin,k),k); }
  function close(bin, k) { return erode(dilate(bin,k),k); }
  function invert(bin) { return bin.map(row => row.map(v => v^1)); }

  // ---- Detecção de contornos via flood-fill com rastreamento de bounding box ----
  // Replica cv2.findContours(img, RETR_EXTERNAL) + filtro is_square_like(tol=0.3)
  // seguido da seleção do maior candidato por área de bounding box (igual ao
  // algoritmo Python de referência).
  function findSquareContours(bin) {
    const H = bin.length, W = bin[0].length;
    const visited = Array.from({length:H}, () => new Uint8Array(W));
    const comps = [];
    for (let sr=0; sr<H; sr++) for (let sc=0; sc<W; sc++) {
      if (!bin[sr][sc] || visited[sr][sc]) continue;
      const q=[[sr,sc]]; visited[sr][sc]=1;
      let minr=sr,maxr=sr,minc=sc,maxc=sc,area=0;
      while(q.length) {
        const [r,c]=q.shift(); area++;
        minr=Math.min(minr,r); maxr=Math.max(maxr,r);
        minc=Math.min(minc,c); maxc=Math.max(maxc,c);
        // 8-conectividade, como o OpenCV usa para contornos
        for (const [dr,dc] of [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]]) {
          const nr=r+dr,nc=c+dc;
          if(nr>=0&&nr<H&&nc>=0&&nc<W&&!visited[nr][nc]&&bin[nr][nc])
            { visited[nr][nc]=1; q.push([nr,nc]); }
        }
      }
      comps.push({minr,maxr,minc,maxc,area});
    }

    // Filtro: aspect ratio da bounding box entre 0.7 e 1.3 (tol=0.3)
    const isSquareLike = (c) => {
      const w = c.maxc-c.minc+1, h = c.maxr-c.minr+1;
      const ratio = h>0 ? w/h : 0;
      return ratio >= 0.7 && ratio <= 1.3;
    };
    const candidatos = comps.filter(isSquareLike);
    return { total: comps.length, candidatos };
  }

  function pickLargestByBBoxArea(candidatos) {
    if (!candidatos.length) return null;
    return candidatos.reduce((best, c) => {
      const areaC = (c.maxc-c.minc+1)*(c.maxr-c.minr+1);
      const areaB = best ? (best.maxc-best.minc+1)*(best.maxr-best.minr+1) : -1;
      return areaC > areaB ? c : best;
    }, null);
  }

  function expandWithMargin(bbox, margem, H, W) {
    return {
      minr: Math.max(bbox.minr - margem, 0),
      minc: Math.max(bbox.minc - margem, 0),
      maxr: Math.min(bbox.maxr + margem, H-1),
      maxc: Math.min(bbox.maxc + margem, W-1)
    };
  }

  function renderBin(ctx, bin, W, H, invert=false, highlight=null, highlightLabel='') {
    const imgData = ctx.createImageData(W, H);
    const scaleR = SZ/H, scaleC = SZ/W;
    for (let py=0; py<H; py++) for (let px=0; px<W; px++) {
      const r=Math.min(SZ-1,Math.floor(py*scaleR));
      const c=Math.min(SZ-1,Math.floor(px*scaleC));
      const v = bin[r][c] ^ (invert?1:0);
      const i=(py*W+px)*4;
      imgData.data[i]  = v?255:0;
      imgData.data[i+1]= v?255:0;
      imgData.data[i+2]= v?255:0;
      imgData.data[i+3]= 255;
    }
    ctx.putImageData(imgData,0,0);
    if (highlight) {
      const s = DISP/SZ;
      ctx.save();
      ctx.strokeStyle='#dc2626'; ctx.lineWidth=2; ctx.setLineDash([5,3]);
      ctx.strokeRect(highlight.minc*s, highlight.minr*s,
                     (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.setLineDash([]);
      ctx.fillStyle='rgba(220,38,38,0.10)';
      ctx.fillRect(highlight.minc*s, highlight.minr*s,
                   (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.font='bold 11px sans-serif'; ctx.fillStyle='#dc2626'; ctx.textAlign='left';
      ctx.fillText(highlightLabel || 'Maior contorno quadrado', highlight.minc*s+3, Math.max(highlight.minr*s-4,10));
      ctx.restore();
    }
  }

  function renderCrop(originalBin, bbox) {
    ctxC.clearRect(0,0,160,160);
    ctxC.fillStyle='#f9fafb'; ctxC.fillRect(0,0,160,160);
    if (!bbox) {
      ctxC.fillStyle='#9ca3af'; ctxC.font='11px sans-serif';
      ctxC.textAlign='center';
      ctxC.fillText('Disponível na', 80, 72);
      ctxC.fillText('etapa 5 · Recorte', 80, 88);
      return;
    }
    const cw=bbox.maxc-bbox.minc+1, ch=bbox.maxr-bbox.minr+1;
    const sc=Math.min(148/cw, 148/ch);
    const dw=Math.round(cw*sc), dh=Math.round(ch*sc);
    const ox=Math.round((160-dw)/2), oy=Math.round((160-dh)/2);
    const imgData=ctxC.createImageData(dw,dh);
    for(let py=0;py<dh;py++) for(let px=0;px<dw;px++) {
      const r=Math.min(SZ-1, bbox.minr+Math.round(py/sc));
      const c=Math.min(SZ-1, bbox.minc+Math.round(px/sc));
      const v=originalBin[r][c]?0:255;
      const i=(py*dw+px)*4;
      imgData.data[i]=v; imgData.data[i+1]=v; imgData.data[i+2]=v; imgData.data[i+3]=255;
    }
    ctxC.putImageData(imgData,ox,oy);
    ctxC.strokeStyle='#dc2626'; ctxC.lineWidth=2;
    ctxC.strokeRect(ox,oy,dw,dh);
  }

  const DESCS = [
    "Imagem original em escala de cinza: documento com QRCode no canto superior direito e bolhas de respostas.",
    "Binarização (mm.threshold): Separa os módulos escuros (QRCode, texto, bolhas) do fundo claro da folha.",
    "Fechamento morfológica (sebox(k)): Ruídos finos menores que k px são juntados, se próximos, pela dilatação seguida de erosão.",
    "cv2.findContours(RETR_EXTERNAL) + filtro de proporção quadrada (0.7–1.3): seleciona o maior contorno candidato.",
    "Recorte final com margem de 5px a partir da imagem original em escala de cinza, na bounding box do contorno selecionado."
  ];

  let curK = 0;  // sebox(0) = 3×3 padrão
  let cachedOpened = {};

  function getProcessed(k) {
    if (!BIN) return null;
    if (!cachedOpened[k]) {
      // mm.sebox(k) = kernel (2k+3)×(2k+3): sebox(0)=3x3, sebox(1)=5x5...
      const kKernel = k + 1;  // mapeia sebox(k) para raio do kernel quadrado
      cachedOpened[k] = close(BIN, kKernel);
    }
    return cachedOpened[k];
  }

  function render() {
    if (!BIN) return;
    const k = curK;
    const opened  = getProcessed(k);
    const inverted = opened; //invert(opened);

    const { total, candidatos } = findSquareContours(inverted);
    const maiorContorno = pickLargestByBBoxArea(candidatos);
    const bboxComMargem = maiorContorno ? expandWithMargin(maiorContorno, 5, SZ, SZ) : null;

    const warnBox = document.getElementById('qr_warnBox');
    if (maiorContorno) {
      const w = maiorContorno.maxc - maiorContorno.minc + 1;
      const h = maiorContorno.maxr - maiorContorno.minr + 1;
      const ratio = w / h;
      if (ratio > 1.5 || ratio < 0.67) {
        warnBox.innerHTML = '<div class="qr_warn">⚠️ O maior contorno quadrado detectado incorporou uma linha horizontal da folha, distorcendo a bounding box. Isso ocorre porque sebox(' + k + ') funde módulos do QRCode com elementos adjacentes. Na prática, o algoritmo MCTest usa critérios adicionais (área mínima e proximidade com a borda superior) para evitar essa situação. Reduza k para obter o recorte correto.</div>';
      } else {
        warnBox.innerHTML = '';
      }
    } else {
      warnBox.innerHTML = '<div class="qr_warn">⚠️ Nenhum contorno quadrado encontrado. Reduza k.</div>';
    }

    const kDescs = [
      'sebox(0) = 3×3 — fechamento leve, preserva módulos finos',
      'sebox(1) = 5×5 — fechamento médio, ideal para segmentar o QRCode',
      'sebox(2) = 7×7 — começa a fundir módulos adjacentes',
      'sebox(3) = 9×9 — destrói a estrutura do QRCode'
    ];
    document.getElementById('qr_kDesc').textContent = kDescs[k] || '';

    cvMain.width = DISP; cvMain.height = DISP;

    switch(curStep) {
      case 0:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 1:
        renderBin(ctxM, BIN, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 2:
        renderBin(ctxM, opened, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 3:
        renderBin(ctxM, inverted, DISP, DISP, false, maiorContorno, `Contorno (${candidatos.length}/${total} quadrado-like)`);
        renderCrop(BIN, null);
        break;
      case 4:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        if (bboxComMargem) {
          const s = DISP/SZ;
          ctxM.save();
          ctxM.strokeStyle = '#dc2626';  // era '#6366f1'
          ctxM.lineWidth = 2;
          ctxM.setLineDash([5,3]);
          ctxM.strokeRect(bboxComMargem.minc*s, bboxComMargem.minr*s,
                          (bboxComMargem.maxc-bboxComMargem.minc)*s,
                          (bboxComMargem.maxr-bboxComMargem.minr)*s);
          ctxM.setLineDash([]);
          ctxM.restore();
        }
        renderCrop(BIN, bboxComMargem);
        break;
    }
  }

  function setStep(s) {
    curStep = s;
    document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((b,i) => b.classList.toggle('active', i===s));
    document.getElementById('qr_main_label').textContent = ['0 · Original','1 · Limiarização','2 · Fechamento','3 · Contorno','4 · Recorte'][s];
    document.getElementById('qr_desc').textContent = DESCS[s];
    document.getElementById('qr_kRow').classList.toggle('visible', s===2);
    render();
  }

  document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((btn,i) => btn.addEventListener('click', ()=>setStep(i)));

  document.getElementById('qr_kSlider').addEventListener('input', function() {
    curK = parseInt(this.value);
    document.getElementById('qr_kVal').textContent = curK;
    render();
  });
}());
</script>
</div>
""")

**Figure 6.11:** Simulateur interactif du *pipeline* d


<figure id="fig-06-sim-06-qrcode">
  <img src="imagens/fig-06-sim-06-qrcode.png" alt=" Simulateur interactif du *pipeline* d'isolation du QRCode : parcourez les étapes de filtrage, ajustez l'élément structurant de la fermeture morphologique et voyez la détection du contour carré sur l'image originale du gabarit. " style="max-width:80%" />
  <figcaption><strong>Figure 6.11:</strong>  Simulateur interactif du *pipeline* d'isolation du QRCode : parcourez les étapes de filtrage, ajustez l'élément structurant de la fermeture morphologique et voyez la détection du contour carré sur l'image originale du gabarit. </figcaption>
</figure>

> ### 📝 🧠 Pourquoi cela fonctionne-t-il ? — Isolement et décodage du *QRCode*
>
> **Ouverture morphologique :** contrairement à la fermeture, l'ouverture (érosion
> suivie d'une dilatation) élimine les petits bruits et protubérances sans modifier
> significativement la géométrie des objets plus grands. Ainsi, elle préserve la
> structure du *QRCode* tout en supprimant les composants parasites qui pourraient
> entraver sa localisation.
>
> **Sélection par géométrie :** le *QRCode* possède une forme approximativement
> carrée ($w/h \approx 1$). La combinaison de ce critère avec la sélection du
> composant de plus grande aire écarte les lignes du formulaire, les textes et
> autres éléments imprimés, permettant d'isoler le marqueur sans recourir à des
> modèles d'apprentissage.
>
> **Redimensionnement avant décodage :** lorsque le *QRCode* occupe peu de
> pixels dans l'image, ses modules deviennent difficiles à distinguer. Le
> redimensionnement avec interpolation cubique augmente la résolution spatiale de
> la région d'intérêt, facilitant l'identification des motifs du code par le
> `cv2.QRCodeDetector` et rendant le décodage plus robuste.

### 6.8.7 Décodage de codes-barres

Dans la section précédente, le décodage des *QRCodes* a été réalisé à l'aide du détecteur natif d'OpenCV (`cv2.QRCodeDetector`), qui intègre dans une interface unique la détection géométrique du symbole, sa rectification et l'extraction de l'information codée.

Pour les codes-barres linéaires (1D), tels que EAN-13, Code 39 et Code 128, une alternative largement utilisée est la bibliothèque `pyzbar`. Contrairement au `QRCodeDetector`, elle prend en charge diverses symbologies de codes-barres et peut également être employée pour la lecture de *QRCodes*.

La procédure consiste à localiser automatiquement chaque symbole présent dans l'image et à interpréter la séquence de barres et d'espaces correspondante, produisant ainsi la chaîne de caractères codée. Outre les données décodées, la bibliothèque fournit des informations telles que la symbologie identifiée et la position du code dans l'image, permettant sa validation ou son traitement ultérieur.

La [Figure 6.12](#fig-06-barcode-decode) présente un exemple de code-barres et le résultat de son décodage à l'aide de la bibliothèque `pyzbar`.

In [15]:
import os
import urllib.request
import numpy as np
from pyzbar.pyzbar import decode
from morph import mm

barcode_path = 'dados/barcode.png'
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/barcode.png"
)

# Si le fichier n'existe pas localement, télécharge automatiquement depuis GitHub
if not os.path.exists(barcode_path):
    print(f"[TÉLÉCHARGEMENT] Téléchargement du code-barres depuis GitHub : {url_github}")
    try:
        os.makedirs(os.path.dirname(barcode_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, barcode_path)
        print("[TÉLÉCHARGEMENT] Image téléchargée avec succès !")
    except Exception as e:
        print(f"[TÉLÉCHARGEMENT] Échec du téléchargement du fichier : {e}")

if os.path.exists(barcode_path):
    image = mm.read(barcode_path)
else:
    # Repli synthétique : génère un motif de barres verticales simulant un Code-128
    print("[AVERTISSEMENT] Fichier 'données/barcode.png' introuvable.")
    print("        Utilisation d'une image synthétique pour la démonstration du pipeline.")
    h, w = 100, 400
    img_synth = np.ones((h, w), dtype=np.uint8) * 255
    # Barres sombres à positions régulières (motif simplifié)
    for x in range(20, w - 20, 8):
        if (x // 8) % 3 != 0:
            img_synth[:, x:x+4] = 0
    image = img_synth

mm.show(image)

# Exécute le décodage avec pyzbar
try:
    barcodes = decode(image)
except ImportError:
    print("[ERREUR] zbar du système introuvable. Exécutez : !apt-get install -y libzbar0")
    barcodes = []

if barcodes:
    dados_bc = barcodes[0].data.decode("utf-8")
    tipo = barcodes[0].type
    print(f"Code-barres décodé avec succès [{tipo}]:\n{dados_bc}")
else:
    print("[INFO] Aucun code-barres détecté dans l'image.")
    print(
        "Sur une image synthétique, c'est attendu — "
        "remplacez par le fichier réel pour décoder."
    )

<Figure size 450x450 with 1 Axes>

**Figure 6.12:** Décodage de code-barres linéaire avec *pyzbar* : image d


Code-barres décodé avec succès [EAN13]:
0000000000055


## 6.9 Le MCTest comme étude de cas : du prototype au système en production

Jusqu’à présent, les principales étapes du *pipeline* de traitement ont été présentées et analysées individuellement, notamment la correction de l’inclinaison (*deskew*), la détection des marqueurs, la rectification par perspective et la lecture des *QRcodes*. Bien que cette approche facilite la compréhension de chaque technique, les applications réelles exigent l’intégration de ces étapes dans un flux de traitement unique et cohérent.

Le **MCTest** constitue un exemple de cette intégration. Développé à l’UFABC et disponible en tant que logiciel open source, le système est utilisé depuis 2012 pour la correction automatisée d’évaluations, offrant un support pour différents modèles de feuilles de réponses, des corrigés individualisés et la génération automatique de rapports de performance (ZAMPIROLLI, 2023).

À partir de ce point, l’accent n’est plus mis sur l’implémentation isolée des algorithmes, mais sur l’organisation de ces algorithmes dans une application complète. Outre la qualité des méthodes de traitement d’images, un système de cette nature doit répondre à des exigences telles que la robustesse face à différentes conditions d’acquisition, la facilité de maintenance et la capacité d’évolution vers de nouvelles fonctionnalités.

Dans les sections suivantes, le module de Vision par Ordinateur du MCTest, implémenté dans le fichier `CVMCTest.py`, sera analysé. L’objectif est de montrer comment les concepts présentés tout au long de ce chapitre sont combinés dans un *pipeline* de traitement utilisé dans une application réelle.

### 6.9.1 Obtention et préparation du module

Le fichier `CVMCTest.py` intègre le système MCTest et, dans sa version originale, dépend de modèles, de configurations et d'autres composants du *framework* Django. Comme ces dépendances ne sont pas disponibles dans l'environnement utilisé dans ce chapitre, le module doit être adapté pour être exécuté de manière indépendante.

Pour ce faire, le fichier est obtenu directement depuis le dépôt du projet à l'aide de la bibliothèque `requests`. Ensuite, des commandes `sed` sont employées pour supprimer les importations et les dépendances spécifiques à l'environnement Web, produisant ainsi une version autonome du module. Cette adaptation préserve l'implémentation des algorithmes de vision par ordinateur, permettant leur exécution et leur analyse sans avoir à installer ou configurer toute l'infrastructure du système MCTest.

In [16]:
import requests
CVMCTest = requests.get(
    "https://raw.githubusercontent.com/fzampirolli/mctest/master/exam/CVMCTest.py"
    )
with open('CVMCTest.py', 'w') as writefile:
    writefile.write(CVMCTest.text)

Chaque commande `sed` supprime les importations spécifiques à l’environnement Django,
rendant le fichier `CVMCTest.py` utilisable de manière indépendante dans ce chapitre.

In [17]:
# supprimer les lignes avec "form django.", ...
!sed --in-place '/from django./d' CVMCTest.py
!sed --in-place '/from exam./d' CVMCTest.py
!sed --in-place '/from mctest./d' CVMCTest.py
!sed --in-place '/from student./d' CVMCTest.py
!sed --in-place '/from topic./d' CVMCTest.py
!sed --in-place '/from .models import VariationExam/d' CVMCTest.py

> ### 📝 🧠 Pourquoi supprimer les dépendances de Django ?
>
> Dans l'implémentation originale, le fichier `CVMCTest.py` fait partie d'une application développée avec le *framework* Django et, de ce fait, importe des modèles, des configurations et d'autres composants spécifiques à cet environnement. Étant donné que ces éléments ne sont pas disponibles dans ce chapitre, le module ne peut pas être importé directement.
>
> Les commandes `sed` suppriment uniquement ces dépendances, sans modifier les routines de Vision par Ordinateur implémentées dans le fichier. Ainsi, le module peut être exécuté de manière indépendante, en préservant le comportement des algorithmes présentés.
>
> Cette procédure illustre un principe important de l'ingénierie logicielle : séparer la logique de l'application de l'infrastructure dans laquelle elle est intégrée, ce qui facilite la réutilisation, les tests et l'étude de composants spécifiques.

### 6.9.2 Extraction de la zone de réponses

Après la lecture de la feuille, la fonction `getAnswerArea` exécute automatiquement les étapes de détection des marqueurs de référence et de redressement par perspective présentées dans les sections précédentes. En conséquence, on obtient une image contenant uniquement la région destinée aux réponses, alignée et aux dimensions standardisées.

Cette standardisation simplifie les étapes ultérieures de traitement, car la localisation des champs de marquage devient connue et indépendante de la position originale de la feuille lors de la numérisation.

La [Figure 6.13](#fig-06-mctest-img-original) présente la feuille de réponses originale en niveaux de gris, tandis que la [Figure 6.14](#fig-06-mctest-answer-area) montre la région de réponses obtenue après l’application de la fonction `getAnswerArea`.

In [18]:
import os
from pdf2image import convert_from_path
from skimage import data as skdata
import cv2
from morph import mm

file = "dados/provas_qrcode_EP.pdf"
MYFILES = 'extra02.qrcode'

if os.path.exists(file):
    pages = convert_from_path(file, 200)  # dpi 100=min 500=max
    numPAGES = 0
    for page in pages:
        myfile0 = MYFILES + '_p' + str(numPAGES) + '.png'
        page.save(myfile0)
        numPAGES += 1
        print(f"[INGESTION] Page convertie : {myfile0}")
    pages.clear()
    img_color = mm.read(myfile0)
    img_inicial = mm.gray(img_color)
else:
    print("[AVERTISSEMENT] Fichier 'dados/provas_qrcode.pdf' introuvable.")
    print("        Utilisation de l'image publique skimage.data.page() comme substitut.")
    img_inicial = skdata.page()

mm.show(img_inicial)


[INGESTION] Page convertie : extra02.qrcode_p0.png


[INGESTION] Page convertie : extra02.qrcode_p1.png


[INGESTION] Page convertie : extra02.qrcode_p2.png


<Figure size 826.5x1169.5 with 1 Axes>

**Figure 6.13:** Image de la feuille de réponses en niveaux de gris chargée à partir du PDF rasterisé.


In [19]:
import CVMCTest
countPage = 0
img_getAnswerArea = CVMCTest.cvMCTest.getAnswerArea(img_inicial, countPage)
mm.show(img_getAnswerArea)

<Figure size 563x511 with 1 Axes>

**Figure 6.14:** Zone de réponses extraite par *getAnswerArea* : région rectifiée contenant les cases de marquage.


**Note de compatibilité :** les versions récentes de NumPy (≥ 2.0) ont supprimé l'alias `np.int0`. Si `CVMCTest.py` utilise ce type, la commande ci-dessous applique la correction directement dans le fichier avant de le recharger :

In [20]:
!sed -i 's/box = np.int0(cv2.boxPoints(rect))/box = cv2.boxPoints(rect).astype(np.intp)/' \
    ./CVMCTest.py

In [21]:
import importlib
import CVMCTest

importlib.reload(CVMCTest)

<module 'CVMCTest' from '/home/fz/VSCode/pdi-vc/gen/quarto/py.fr/cap06/CVMCTest.py'>

### 6.9.3 Segmentation du *QRCode*

Après l’extraction de la zone de réponses, MCTest effectue deux étapes préparatoires à la lecture des marquages : la segmentation du *QRCode* et la localisation des cadres contenant les questions.

La fonction `segmentQRcode` isole la région de l’image correspondant au *QRCode*, présentée dans la [Figure 6.15](#fig-06-mctest-qrcode-seg2). Ensuite, cette région est traitée par la fonction `getQRCode`, chargée de son décodage et de l’extraction des métadonnées de l’épreuve.

Si le *QRCode* ne peut pas être décodé, le traitement de la feuille se poursuit normalement. Les réponses de l’étudiant sont toujours lues et enregistrées dans le fichier CSV de sortie ; seules les informations obtenues à partir du *QRCode*, telles que l’identification de l’épreuve ou de l’étudiant, restent indisponibles.

In [22]:
import CVMCTest
imgQRcode = CVMCTest.cvMCTest.segmentQRcode(img_getAnswerArea, countPage)
mm.show(imgQRcode)

<Figure size 450x450 with 1 Axes>

**Figure 6.15:** Région du *QRCode* isolée par *segmentQRcode* dans la zone de réponses rectifiée.


La fonction `CVMCTest.cvMCTest.getQRCode(img, countPage)` intègre les étapes de segmentation et de décodage du *QRCode*. Internement, elle utilise `CVMCTest.cvMCTest.decodeQRcode(imgQRcode)` pour interpréter la *chaîne* hexadécimale encodée dans le symbole et construire le dictionnaire `qr`, en plus de retourner l'indicateur logique `myFlagArea`, qui informe si la lecture a été réalisée avec succès.

Le dictionnaire `qr` regroupe les métadonnées de l'épreuve utilisées dans les étapes ultérieures du traitement. Ses principaux champs sont :

- **`date` :** identifiant temporel de l'épreuve, composé de la date de génération et d'un *timestamp* interne du MCTest.
- **`idClassroom`, `idExam` et `idStudent` :** identifiants de la classe, de l'épreuve et de l'étudiant.
- **`term` :** période scolaire.
- **`stylesheet` :** feuille de style utilisée lors de la génération du formulaire.
- **`var1` à `var5` :** nombre de questions à chaque niveau de difficulté.
- **`text` :** nombre de questions dissertatives.
- **`answer` :** nombre d'alternatives par question.
- **`numquest` :** nombre total de questions.
- **`correct`** et **`dbtext` :** champs remplis lors de la correction, contenant le corrigé et des informations supplémentaires.
- **`variations`** et **`variant` :** informations sur les versions de l'épreuve.

Ces métadonnées identifient l'épreuve et l'étudiant, permettant de sélectionner le corrigé correspondant et de paramétrer les étapes suivantes de lecture et de correction des réponses.

In [23]:
myFlagArea, qr = CVMCTest.cvMCTest.getQRCode(img_inicial, countPage)
myFlagArea, qr

(True,
 {'date': '260208-1770403541363',
  'idClassroom': '955',
  'idExam': '810',
  'idStudent': '448898',
  'term': '0',
  'stylesheet': '1',
  'var1': '50',
  'var2': '0',
  'var3': '0',
  'var4': '0',
  'var5': '0',
  'text': '0',
  'answer': '5',
  'numquest': 50,
  'correct': '',
  'dbtext': '',
  'variations': '0',
  'variant': '0'})

Ces métadonnées permettent d'identifier l'épreuve, de récupérer le corrigé correspondant et de paramétrer les étapes ultérieures du traitement.

Après le décodage du *QRCode*, le traitement revient à la zone des réponses pour localiser les cadres contenant les marquages de l'étudiant.

### 6.9.4 Localisation des cadres de réponses

L’image produite par `getAnswerArea` contient toute la région utile de la feuille, y compris l’en-tête, où se trouve le *QRCode*, ainsi que les cadres destinés aux réponses. Comme la lecture des marquages n’utilise que ces cadres, la région correspondant à l’en-tête est éliminée au moyen du recadrage `img_getAnswerArea[300:, :]`.

La [Figure 6.16](#fig-06-mctest-answer-area-crop) présente cette région d’intérêt. Bien que ce recadrage soit par la suite utilisé par la fonction `findSquares` pour localiser les cadres de réponses, MCTest effectue initialement la lecture du *QRCode*, car celui-ci contient les métadonnées nécessaires pour identifier l’épreuve et configurer les étapes ultérieures du traitement.

In [24]:
img_getAnswerArea_aux = img_getAnswerArea[300:,:]
mm.show(img_getAnswerArea_aux)

<Figure size 563x450 with 1 Axes>

**Figure 6.16:** Recorte inférieur de la zone de réponses, concentrant les cadres de bulles à segmenter.


La fonction `findSquares` reçoit l'image de la zone de réponses et les métadonnées stockées dans `qr`, et retourne les coordonnées des cadres qui délimitent les groupes de questions.

Chaque élément de `rectSquares` contient les coordonnées des sommets supérieur gauche et inférieur droit d'un cadre de réponses, qui seront utilisées dans l'étape de segmentation des bulles.

In [25]:
rectSquares = CVMCTest.cvMCTest.findSquares(qr,img_getAnswerArea, countPage)
rectSquares

[[[np.int64(395), np.int64(350)], [np.int64(950), np.int64(516)]],
 [[np.int64(394), np.int64(602)], [np.int64(951), np.int64(767)]]]

### 6.9.5 Lecture automatique des réponses

Une fois les métadonnées de l’épreuve (`qr`) et les coordonnées des cadres de réponses (`rectSquares`) connues, le MCTest identifie automatiquement les alternatives cochées par l’étudiant.

Le code suivant intègre les étapes présentées précédemment. Pour chaque cadre délimité dans `rectSquares`, les fonctions `setColumns` et `setLines` estiment respectivement le nombre d’alternatives par question et le nombre de questions à partir de la distribution spatiale des bulles. Ensuite, `segmentAnswers` détermine l’alternative sélectionnée pour chaque question, et `setAnswersOneLine` regroupe les résultats de tous les cadres dans le champ `qr['answers']`.

Dans le mode de fonctionnement adopté dans ce chapitre, où le MCTest est exécuté indépendamment de sa base de données, le contenu de `qr['answers']` est comparé au corrigé stocké dans la première page du fichier PDF, correspondant au modèle d’épreuve sans énoncés utilisé dans les exemples.

La [Figure 6.17](#fig-06-mctest-answers) présente les cadres de réponses traités par l’algorithme, tandis que la sortie du programme affiche le contenu final de `qr['answers']`.

In [26]:
testAnswers = []
if myFlagArea:
  
  imgQ_all = []

  for countSquare in range(len(rectSquares)):
      p1, p2 = rectSquares[countSquare]

      if True:
          imgQi = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
          [NUM_COLUMNS, img] = CVMCTest.cvMCTest.setColumns(imgQi, countPage, countSquare)
          [NUM_LINES, img] = CVMCTest.cvMCTest.setLines(imgQi, countPage, countSquare)
          NUM_RESPOSTAS = NUM_COLUMNS
          NUM_QUESTOES = NUM_LINES

      imgQiNC = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
      testAnswers.append(CVMCTest.cvMCTest.segmentAnswers(
          [imgQi, imgQiNC], countPage, countSquare, NUM_QUESTOES, qr

      ))

      imgQ_all.append(imgQiNC)

  qr = CVMCTest.cvMCTest.setAnswarsOneLine(testAnswers, qr)  
  # met les réponses de chaque tableau sur une ligne

mm.show(imgQ_all)
print(f"Réponses lues des {len(qr['answers'].split(","))} questions : \
      \n{qr['answers'][:-19]}...")

<Figure size 2250x750 with 3 Axes>

**Figure 6.17:** Réponses lues automatiquement par MCTest après segmentation et classification de toutes les bulles.


Réponses lues des 50 questions :       
C,A,A,C,E,D,E,C,A,E,B,B,A,D,A,B,C,B,E,D,B,D,A,C,E,B,A,A,B,B,C,A,C,A,C,A,B,C,C,C,...


> ### 💡 Relier les points
>
> Le champ `qr['answers']` représente le résultat final du *pipeline* de vision par ordinateur présenté dans ce chapitre. Son obtention intègre toutes les étapes étudiées, depuis la rastérisation du document et la rectification géométrique jusqu'à l'extraction de la zone de réponses, le décodage du *QRCode*, la localisation des cadres et l'identification des alternatives marquées.
>
> Ce *pipeline* illustre la transition d'un prototype vers un système en production. Dans MCTest, les algorithmes de vision par ordinateur restent essentiellement les mêmes ; les principales différences se concentrent sur des aspects d'ingénierie logicielle, tels que le traitement des exceptions, le support de différents modèles de formulaires, l'intégration avec la base de données, l'interface Web et les mécanismes d'audit et de maintenance.
>
> Dans les expériences de ce chapitre, la première page du fichier PDF contient le corrigé de l'examen, tandis que les pages suivantes correspondent aux feuilles de réponses des étudiants. Après l'obtention de `qr['answers']`, MCTest compare automatiquement les réponses lues avec le corrigé pour calculer le score de chaque étudiant.
>
> Lors de l'utilisation complète du système, les résultats de la correction sont consolidés dans un fichier CSV et envoyés à l'enseignant accompagnés d'un fichier compressé contenant des informations auxiliaires pour l'audit. Parmi ces fichiers figurent les extraits des questions où des marquages multiples ou d'autres situations nécessitant une révision manuelle ont été détectés. L'enseignant peut alors inspecter ces images, décider de l'interprétation la plus appropriée et, si nécessaire, mettre à jour le fichier CSV avant l'importation définitive des notes.

## 6.10 Inspection industrielle automatisée

L'inspection visuelle automatisée est une application de la vision par ordinateur dans laquelle des images de pièces ou de produits sont analysées pour vérifier le respect de critères de qualité préalablement définis. Dans une ligne de production, les images peuvent être obtenues par des caméras ou d'autres dispositifs d'acquisition et traitées automatiquement pour identifier des défauts, mesurer des dimensions ou vérifier la présence de composants.

La stratégie d'inspection dépend des caractéristiques du produit, du type de défaut d'intérêt et de la disponibilité d'une image de référence. Dans ce chapitre, deux approches classiques sont présentées :

- **Soustraction d'images :** compare l'image de la pièce inspectée avec une image de référence considérée comme exempte de défauts. Les régions où la différence d'intensité dépasse un seuil sont classées comme défauts possibles. Cette approche suppose que les images soient géométriquement alignées et aient été acquises dans des conditions d'éclairage similaires.

- **Analyse de texture :** utilise les caractéristiques de la texture de la surface pour identifier les régions dont l'apparence diffère du motif attendu, sans nécessiter d'image de référence. Cette approche est adaptée aux matériaux présentant une texture approximativement homogène, comme les tissus, les papiers et les surfaces métalliques.

Dans les sections suivantes, ces deux stratégies sont illustrées au moyen d'exemples construits à partir d'images de la bibliothèque `skimage.data`. L'objectif est de présenter les principes de fonctionnement de chaque approche dans des expériences pouvant être reproduites intégralement par le lecteur.

### 6.10.1 Références et *Datasets* publics

Dans les applications d'inspection industrielle, la performance des algorithmes de détection de défauts est souvent évaluée sur des *datasets* publics, qui fournissent des images représentatives et, dans de nombreux cas, des annotations de référence (*ground truth*). Dans ce chapitre, cependant, les exemples utilisent des images synthétiques dérivées de `skimage.data` (voir [Figure 6.18](#fig-06-industrial-defeito)), permettant de reproduire toutes les expériences sans dépendance à des bases de données externes.

Pour des études plus complètes et la comparaison entre algorithmes, on distingue les *datasets* publics suivants :

- **MVTec *Anomaly Detection Dataset* (MVTec AD) :** ensemble d'images d'objets et de textures, contenant des échantillons sans défauts et avec défauts, accompagnés de masques de segmentation au niveau du pixel pour les images anormales (BERGMANN, 2019). Disponible sur : <https://www.mvtec.com/company/research/datasets/mvtec-ad>.

- **Kolektor *Surface-Defect Dataset* (KolektorSDD) :** ensemble d'images de composants industriels avec des défauts de surface annotés, utilisé dans les études de détection et de segmentation de défauts (TABERNIK, 2020). Disponible sur : <https://www.vicos.si/resources/kolektorsdd/>.

- **NEU *Surface Defect Database* :** ensemble d'images de surfaces d'acier laminé, organisé en six catégories de défauts de surface, fréquemment utilisé dans l'évaluation des méthodes de classification et de détection (SONG, 2013). Disponible sur : <http://faculty.neu.edu.cn/songkechen/zh_CN/zdylm/263270/list/index.htm>.

In [27]:
import numpy as np
from skimage import data, color
from morph import mm

# Image de référence (produit sans défaut)
product_color = data.coffee()
product_gray = color.rgb2gray(product_color)

# Insertion de défaut simulé : rayure sombre de 10×100 px
defect_image = np.copy(product_gray)
defect_image[100:110, 200:300] = 0.1

# Détection par soustraction et seuillage
difference = np.abs(product_gray - defect_image)
defect_threshold = 0.15          # ajustable selon l'application
detected_defect = (difference > defect_threshold).astype(np.uint8) * 255

mm.show(
    [product_gray, defect_image, detected_defect],
    titles=["Référence", "Avec défaut", "Défaut détecté"],
    cols=3,
    figsize=(12, 4)
)

status = "Defeito detectado." if detected_defect.any() else "Produto conforme."
print(status)

<Figure size 1800x600 with 3 Axes>

**Figure 6.18:** Détection de défaut par soustraction d


Defeito detectado.


> ### 📝 Soustraction d'images : enregistrement géométrique et principe de fonctionnement
>
> La soustraction d'images suppose que l'image d'inspection soit géométriquement alignée sur l'image de référence. Les différences de positionnement, de rotation, d'échelle ou de perspective produisent des régions de différence qui peuvent être confondues avec des défauts.
>
> Dans les applications à acquisition contrôlée, cet alignement est obtenu lors de la capture au moyen de gabarits mécaniques, de tapis roulants et de caméras fixes, ce qui permet de comparer directement des images successives. Un exemple est l'inspection d'une boîte à outils toujours positionnée dans la même orientation afin de vérifier l'absence d'un quelconque élément.
>
> Lorsque ce contrôle n'est pas possible, on emploie l'**enregistrement d'images**, qui estime une transformation géométrique pour compenser les différences de translation, de rotation, d'échelle et, si nécessaire, de perspective.
>
> Après l'enregistrement, on effectue la comparaison pixel par pixel entre les deux images. Dans les régions sans modification, les différences d'intensité tendent à être proches de zéro ; là où un défaut existe, des différences locales apparaissent et peuvent être mises en évidence par seuillage. L'utilisation de la différence absolue permet de détecter aussi bien les défauts plus clairs que plus sombres que la référence.
>
> La performance de la méthode dépend principalement de la qualité de l'alignement géométrique et du choix du seuil utilisé pour séparer les petites variations d'acquisition des différences associées aux défauts.
>
> La [Figure 6.19](#fig-06-sim-06-industrial) illustre l'effet du désalignement entre les images et l'importance de l'enregistrement géométrique avant l'application de la soustraction.

In [28]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-industrial" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-industrial * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-industrial canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-industrial button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-industrial button:hover { background: #e8dfcf; }
  #sim-06-industrial button.sim06_ind_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-industrial input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; margin: 4px 0; }
  .sim06_ind_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ind_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ind_grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; }
  .sim06_ind_control_group { background: #ffffff; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; }
  .sim06_ind_label_wrap { display: flex; justify-content: space-between; font-size: 11px; font-weight: 700; color: #5e5a4a; }
  .sim06_ind_val { font-family: monospace; color: #26241d; }
  .sim06_ind_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .sim06_ind_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .sim06_ind_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .sim06_ind_status { font-size: 11px; font-weight: 700; padding: 10px 12px; border-radius: 8px; text-align: center; margin-top: 10px; transition: all 0.2s ease; border: 1px solid transparent; line-height: 1.4; }
  .sim06_ind_status.ok { background: #eafaf1; color: #04342C; border-color: #a3e4d7; }
  .sim06_ind_status.fail { background: #fdecea; color: #4A1B0C; border-color: #f5b7b1; }
  .sim06_ind_status.warn { background: #fef5e7; color: #412402; border-color: #f8c471; }
  .sim06_ind_reg_toggle { display: flex; gap: 6px; margin-top: 8px; }
  .sim06_ind_reg_toggle button { flex: 1; font-weight: 600; justify-content: center; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⚙️ Simulateur : Soustraction d'images et recalage géométrique</span>
  <span class="sim06_ind_pill">|R�férence − Inspection| &gt; Seuil</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ind_panel">
    <div class="sim06_ind_grid">
      
      <div class="sim06_ind_control_group">
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
          🔄 Désalignement de la capture (convoyeur)
        </div>

        <div class="sim06_ind_label_wrap"><span>Translation horizontale (Δx) :</span><span id="sim06_ind_txVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTx" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Translation verticale (Δy) :</span><span id="sim06_ind_tyVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTy" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Rotation (θ) :</span><span id="sim06_ind_rotVal" class="sim06_ind_val">0.0°</span></div>
        <input type="range" id="sim06_ind_sliderRot" min="-5" max="5" step="0.5" value="0">

        <div class="sim06_ind_reg_toggle">
          <button id="sim06_ind_btnModoDireto" class="sim06_ind_active">Soustraction directe</button>
          <button id="sim06_ind_btnModoRegistro">Soustraction avec recalage</button>
        </div>
      </div>

      <div class="sim06_ind_control_group" style="display:flex; flex-direction:column; justify-space-between;">
        <div>
          <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
            🎛️ Paramétrage de l'inspecteur
          </div>
          <div class="sim06_ind_label_wrap"><span>Seuil de tolérance (T) :</span><span id="sim06_ind_thVal" class="sim06_ind_val">35</span></div>
          <input type="range" id="sim06_ind_sliderTh" min="10" max="100" step="5" value="35">
        </div>
        <div style="text-align:right; margin-top:12px;">
          <button id="sim06_ind_btnReset">↺ Réinitialiser la simulation</button>
        </div>
      </div>

    </div>

    <div id="sim06_ind_msgStatus" class="sim06_ind_status ok">Produit conforme.</div>
  </div>

  <!-- Canvases de Exibição -->
  <div class="sim06_ind_canvases" style="margin-top:14px;">
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label">1. Référence stable</div>
      <canvas id="sim06_ind_cvRef" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_inspLabel">2. Inspection (capture réelle)</div>
      <canvas id="sim06_ind_cvInsp" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_diffLabel">3. Masque d'anomalie</div>
      <canvas id="sim06_ind_cvDiff" width="220" height="170"></canvas>
    </div>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_ind_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Défi pédagogique</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      Avec le mode <strong>Soustraction directe</strong> actif, utilisez les contrôles de translation et de rotation pour simuler de légers
      désalignements sur le convoyeur. Observez que des variations de quelques pixels ou degrés génèrent des contours faciles à confondre avec de véritables défauts.
      Augmenter le <strong>seuil de tolérance</strong> pour ignorer ces contours réduit la sensibilité, rendant le système aveugle aux défauts fins (comme l'égratignure sur le côté gauche).
      En basculant sur <strong>Soustraction avec recalage</strong>, l'alignement est restauré avant la différence, isolant avec précision le défaut réel sans fausses alarmes.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Industrial(root){
    if (!root || root.dataset.sim06IndustrialInit) return;
    root.dataset.sim06IndustrialInit = "1";

    const W = 220, H = 170;

    const cvRef  = root.querySelector('#sim06_ind_cvRef');
    const cvInsp = root.querySelector('#sim06_ind_cvInsp');
    const cvDiff = root.querySelector('#sim06_ind_cvDiff');

    const ctxRef  = cvRef.getContext('2d');
    const ctxInsp = cvInsp.getContext('2d');
    const ctxDiff = cvDiff.getContext('2d');

    let modoRegistro = false;

    function drawComponent(ctx, hasDefect, tx, ty, rotDeg) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.save();
      ctx.translate(W / 2 + tx, H / 2 + ty);
      ctx.rotate(rotDeg * Math.PI / 180);

      ctx.fillStyle = '#bdc3c7';
      ctx.strokeStyle = '#2c3e50';
      ctx.lineWidth = 3;

      ctx.beginPath();
      ctx.rect(-60, -45, 120, 90);
      ctx.fill(); ctx.stroke();

      ctx.beginPath();
      ctx.arc(0, 0, 35, 0, Math.PI * 2);
      ctx.fillStyle = '#ecf0f1';
      ctx.fill(); ctx.stroke();

      ctx.fillStyle = '#fafaf7';
      const holes = [[-40, -25], [40, -25], [-40, 25], [40, 25]];
      holes.forEach(([hx, hy]) => {
        ctx.beginPath();
        ctx.arc(hx, hy, 8, 0, Math.PI * 2);
        ctx.fill(); ctx.stroke();
      });

      ctx.beginPath();
      ctx.arc(0, 0, 12, 0, Math.PI * 2);
      ctx.fillStyle = '#2c3e50';
      ctx.fill();

      if (hasDefect) {
        ctx.save();
        ctx.strokeStyle = '#17202a';
        ctx.lineWidth = 3;
        ctx.lineCap = 'round';
        ctx.beginPath();
        ctx.moveTo(-45, 5);
        ctx.lineTo(-25, -15);
        ctx.stroke();
        ctx.restore();
      }

      ctx.restore();
    }

    function update() {
      const tx  = parseInt(root.querySelector('#sim06_ind_sliderTx').value);
      const ty  = parseInt(root.querySelector('#sim06_ind_sliderTy').value);
      const rot = parseFloat(root.querySelector('#sim06_ind_sliderRot').value);
      const th  = parseInt(root.querySelector('#sim06_ind_sliderTh').value);

      root.querySelector('#sim06_ind_txVal').textContent  = (tx > 0 ? '+' : '') + tx + ' px';
      root.querySelector('#sim06_ind_tyVal').textContent  = (ty > 0 ? '+' : '') + ty + ' px';
      root.querySelector('#sim06_ind_rotVal').textContent = (rot > 0 ? '+' : '') + rot.toFixed(1) + '°';
      root.querySelector('#sim06_ind_thVal').textContent  = th;

      drawComponent(ctxRef, false, 0, 0, 0);

      const tx_insp  = modoRegistro ? 0 : tx;
      const ty_insp  = modoRegistro ? 0 : ty;
      const rot_insp = modoRegistro ? 0 : rot;
      drawComponent(ctxInsp, true, tx_insp, ty_insp, rot_insp);

      const dataRef  = ctxRef.getImageData(0, 0, W, H);
      const dataInsp = ctxInsp.getImageData(0, 0, W, H);
      const dataDiff = ctxDiff.createImageData(W, H);

      let defectPixelsDetected = 0;

      for (let i = 0; i < dataRef.data.length; i += 4) {
        const diff = Math.abs(dataRef.data[i] - dataInsp.data[i]);

        if (diff > th) {
          dataDiff.data[i]   = 192; // R
          dataDiff.data[i+1] = 57;  // G
          dataDiff.data[i+2] = 43;  // B (#c0392b)
          dataDiff.data[i+3] = 255;
          defectPixelsDetected++;
        } else {
          dataDiff.data[i]   = 250;
          dataDiff.data[i+1] = 250;
          dataDiff.data[i+2] = 247; // #fafaf7
          dataDiff.data[i+3] = 255;
        }
      }
      ctxDiff.putImageData(dataDiff, 0, 0);

      root.querySelector('#sim06_ind_inspLabel').textContent = modoRegistro
        ? '2. Inspeção Registrada (alinhada)'
        : '2. Inspeção (captura real)';
      root.querySelector('#sim06_ind_diffLabel').textContent = modoRegistro
        ? '3. Máscara de Anomalia (com registro)'
        : '3. Máscara de Anomalia (direta)';

      const msgBox = root.querySelector('#sim06_ind_msgStatus');
      const hasMisalignment = (tx !== 0 || ty !== 0 || rot !== 0);
      const misalignmentVisible = hasMisalignment && !modoRegistro;

      if (defectPixelsDetected > 0) {
        if (misalignmentVisible && defectPixelsDetected > 150) {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '❌ Falsos positivos detectados! O desalinhamento mecânico gerou anomalias fantasmas nas bordas (' + defectPixelsDetected + ' px comprometidos).';
        } else if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '⚠️ Anomalia localizada, mas parte da diferença ainda vem do desalinhamento (' + defectPixelsDetected + ' px afetados).';
        } else {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '⚠️ Anomalia localizada! Defeito superficial detectado na estrutura interna (' + defectPixelsDetected + ' px afetados).';
        }
      } else {
        if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '✓ Conforme sob risco. O limiar alto omitiu as bordas desalinhadas, mas tornou o sistema cego para defeitos reais.';
        } else {
          msgBox.className = 'sim06_ind_status ok';
          msgBox.innerHTML = '✓ Produto conforme. Sistema corretamente registrado em nível geométrico.';
        }
      }
    }

    ['#sim06_ind_sliderTx', '#sim06_ind_sliderTy', '#sim06_ind_sliderRot', '#sim06_ind_sliderTh'].forEach(sel => {
      root.querySelector(sel).addEventListener('input', update);
    });

    const btnDireto   = root.querySelector('#sim06_ind_btnModoDireto');
    const btnRegistro = root.querySelector('#sim06_ind_btnModoRegistro');

    btnDireto.addEventListener('click', () => {
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    btnRegistro.addEventListener('click', () => {
      modoRegistro = true;
      btnRegistro.classList.add('sim06_ind_active');
      btnDireto.classList.remove('sim06_ind_active');
      update();
    });

    root.querySelector('#sim06_ind_btnReset').addEventListener('click', function() {
      root.querySelector('#sim06_ind_sliderTx').value = 0;
      root.querySelector('#sim06_ind_sliderTy').value = 0;
      root.querySelector('#sim06_ind_sliderRot').value = 0;
      root.querySelector('#sim06_ind_sliderTh').value = 35;
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    update();
  }

  function tryInitSim06Industrial(){
    var root = document.getElementById('sim-06-industrial');
    if (root) initSim06Industrial(root); else setTimeout(tryInitSim06Industrial, 200);
  }
  tryInitSim06Industrial();
})();
</script>
</div>
""")

**Figure 6.19:** Simulateur interactif d


<figure id="fig-06-sim-06-industrial">
  <img src="imagens/fig-06-sim-06-industrial.png" alt=" Simulateur interactif d'inspection industrielle par soustraction d'images : contrôlez les distorsions géométriques de désalignement (enregistrement) et le seuil de détection pour observer l'impact sur les faux positifs. " style="max-width:80%" />
  <figcaption><strong>Figure 6.19:</strong>  Simulateur interactif d'inspection industrielle par soustraction d'images : contrôlez les distorsions géométriques de désalignement (enregistrement) et le seuil de détection pour observer l'impact sur les faux positifs. </figcaption>
</figure>

### 6.10.2 Détection de défauts par analyse de texture

Dans les applications où il n'existe pas d'image de référence, la détection de défauts peut être basée sur les caractéristiques de texture de la surface. Dans ce cas, on cherche à identifier les régions dont l'apparence diffère du motif prédominant du matériau.

Dans cet exemple, on utilise la **variance locale** comme mesure d'hétérogénéité. Pour chaque position de l'image, on calcule la variance des niveaux d'intensité dans un voisinage de dimensions fixes. Les régions à faible variance tendent à présenter une texture plus uniforme, tandis que des altérations locales, telles que des rayures, des taches ou des imperfections, peuvent produire des valeurs plus élevées de cette mesure.

La [Figure 6.20](#fig-06-industrial-textura) illustre cette procédure en utilisant l'image `skimage.data.brick()`. Dans un premier temps, on calcule la carte de variance locale au moyen d'une fenêtre glissante. Ensuite, on applique un seuillage pour mettre en évidence les régions dont la variance dépasse la valeur spécifiée, identifiant ainsi les zones d'intérêt potentielles pour l'inspection.

#### Modélisation mathématique

Considérons une image en niveaux de gris représentée par

$$
f:\Omega\subset\mathbb{Z}^2\rightarrow\mathbb{R},
$$

où $\Omega$ est le domaine de l'image et $f(x,y)$ représente l'intensité du pixel aux coordonnées $(x,y)$. Dans les images 8 bits, ces intensités appartiennent à l'intervalle $[0,255]$. Dans cet exemple, elles ont toutefois été normalisées dans l'intervalle $[0,1]$, sans modifier le fonctionnement de l'algorithme.

Pour chaque position de l'image, on considère un voisinage carré $W_{x,y}$ de dimension $15\times15$ pixels.

La moyenne locale est donnée par

$$
\mu(x,y)=
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v),
$$

et la variance locale est calculée par

$$
\sigma^2(x,y) =
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v)^2 -
\mu(x,y)^2.
$$

Dans l'implémentation suivante, ces deux moyennes sont obtenues par la fonction `cv2.blur`,

```python
mean  = cv2.blur(img, (15,15))
mean2 = cv2.blur(img**2, (15,15))
var   = mean2 - mean**2
```

Ensuite, on calcule la différence entre les cartes de variance de l'image de référence et de l'image inspectée,

$$
D(x,y)=
\left|
\sigma_d^2(x,y)-\sigma_r^2(x,y)
\right|,
$$

où $\sigma_r^2(x,y)$ et $\sigma_d^2(x,y)$ sont respectivement les variances locales de l'image de référence et de l'image contenant le défaut. Après la normalisation de la carte $D(x,y)$, on applique un seuillage pour obtenir le masque des anomalies possibles.

In [29]:
import numpy as np
import cv2
from skimage import data as skdata
from morph import mm

# Image de texture uniforme (brique)
texture = skdata.brick().astype(np.float32) / 255.0

# Insertion de défaut synthétique : tache claire 20×80 px
texture_defect = np.copy(texture)
texture_defect[60:80, 80:160] = 0.95

# Carte de variance locale (fenêtre 15×15)
def variancia_local(img, ksize=15):
    img_f = img.astype(np.float32)
    mean  = cv2.blur(img_f, (ksize, ksize))
    mean2 = cv2.blur(img_f ** 2, (ksize, ksize))
    return np.clip(mean2 - mean ** 2, 0, None)

var_ref    = variancia_local(texture)
var_defect = variancia_local(texture_defect)
diff_var   = np.abs(var_defect - var_ref)

# Normalise et seuille
diff_norm = (diff_var / diff_var.max() * 255).astype(np.uint8)
_, mask   = cv2.threshold(diff_norm, 30, 255, cv2.THRESH_BINARY)

mm.show(
    [texture, texture_defect, diff_norm, mask],
    titles=["Texture originale", "Avec défaut", "Δ variance locale", "Anomalie détectée"],
    cols=4,
    figsize=(16, 4)
)

status = "Defeito de textura detectado." if mask.any() else "Superfície conforme."
print(status)


<Figure size 2400x600 with 4 Axes>

**Figure 6.20:** Détection d


Defeito de textura detectado.


> ### 📝 🧠 Pourquoi cela fonctionne-t-il ? — Analyse de texture
>
> La variance locale mesure la dispersion des intensités dans un voisinage de l’image. Dans les régions où la texture reste uniforme, cette mesure tend à varier peu. Lorsqu’un défaut modifie le motif de la surface, la distribution des intensités se modifie également, produisant des différences dans la variance locale.
>
> Dans ce chapitre, la détection est réalisée en comparant les cartes de variance de l’image de référence et de l’image avec défaut. Après normalisation, on applique un seuillage pour mettre en évidence les régions où cette différence dépasse une valeur spécifiée.
>
> Les principaux paramètres de la méthode sont la taille de la fenêtre utilisée pour le calcul de la variance et le seuil employé pour la segmentation. Des fenêtres plus petites sont plus sensibles aux détails fins, tandis que des fenêtres plus grandes produisent des cartes plus lisses et peuvent réduire la réponse aux défauts de petites dimensions.

## 6.11 Résumé

Dans ce chapitre, des méthodes de Vision par Ordinateur appliquées à l'analyse de documents et à l'inspection visuelle automatisée ont été présentées. Les principales techniques étudiées étaient :

- **Prétraitement de documents :** application de la normalisation du fond, de l'égalisation adaptative (CLAHE) et du seuillage par Otsu pour réduire les effets d'un éclairage non uniforme et améliorer la segmentation du texte.

- **Reconnaissance optique de caractères (OCR) :** conversion d'images de documents en texte codé au moyen du Tesseract OCR, mettant en évidence l'influence du prétraitement sur la qualité de la reconnaissance.

- **Traduction automatique :** application de techniques de traitement du langage naturel pour traduire le texte obtenu par l'OCR.

- **Redressement géométrique de documents :** utilisation du détecteur de contours de Canny, de la Transformée de Hough et de transformations projectives pour corriger la perspective de documents numérisés.

- **Localisation et redressement de formulaires :** emploi d'opérations morphologiques, d'analyse de contours et de transformation de perspective pour identifier des marqueurs de référence et extraire automatiquement des régions d'intérêt.

- **Lecture de codes bidimensionnels et unidimensionnels :** détection et décodage de *QRCodes* et de codes-barres pour l'identification automatique de documents et de métadonnées.

- **Reconnaissance optique de marques (OMR) :** lecture automatisée de formulaires et de feuilles de réponses, illustrée par une étude de cas du système MCTest.

- **Inspection industrielle :** détection de défauts par comparaison avec une image de référence et par analyse de la variance locale.

Tout au long du chapitre, les algorithmes ont été implémentés et évalués avec des images de la bibliothèque `skimage.data` et avec des documents réels, permettant de reproduire les expériences présentées.

### Prochaines étapes

Les méthodes présentées dans ce chapitre montrent comment des techniques de Traitement Numérique des Images, de Vision par Ordinateur et de Traitement du Langage Naturel peuvent être intégrées dans des *pipelines* pour l'analyse automatisée de documents.

Dans le prochain chapitre, seront étudiées des techniques d'**extraction de caractéristiques** et de **reconnaissance de formes**, avec un accent sur les descripteurs capables de représenter des images par des attributs numériques pour la comparaison, la classification et la reconnaissance automatique. Ces concepts constituent la base des chapitres consacrés à l'apprentissage automatique et à l'apprentissage profond appliqués à la Vision par Ordinateur.

## 6.12 🤖 Utilisation de Gemini Notebook comme tuteur complémentaire

Pour accompagner l'étude de ce chapitre, il est recommandé d'utiliser **Gemini Notebook** comme tuteur complémentaire. L'outil recourt à des modèles d'intelligence artificielle pour répondre aux questions, élaborer des résumés et expliquer des concepts à partir des documents fournis comme source de consultation, permettant à l'étudiant de réviser le contenu de manière interactive.

> ### ❗ 🎓 Étudiez avec le tuteur intelligent
>
> [🚀 ACCÉDER À Gemini Notebook : CHAPITRE 06](https://notebooklm.google.com/notebook/835ec2a8-dbc3-46c4-8c2c-2040af3754a4)
>
> #### 🌐 Langue et langage de programmation
>
> Le projet de ce chapitre dans Gemini Notebook a été construit uniquement avec le texte en **portugais** et les exemples de code en **Python**. Si vous étudiez à partir de l'édition en anglais ou en français, ou que vous suivez le parcours en C++, les réponses du tuteur peuvent ne pas correspondre exactement à la version que vous lisez.
>
> #### ⚠️ Usage critique des réponses
>
> Les réponses générées par Gemini Notebook sont produites automatiquement par un modèle d'intelligence artificielle et peuvent comporter des omissions ou des imprécisions. Pour cette raison, elles doivent être utilisées comme matériel de soutien, et non comme substitut à l'étude du chapitre.
>
> En cas de doute, consultez le texte de cet ouvrage, exécutez les exemples présentés et, si nécessaire, complétez la consultation avec des livres, des articles scientifiques et d'autres sources académiques fiables.

## 6.13 Liste d'exercices

Les exercices suivants consolident les concepts présentés dans ce chapitre à travers des adaptations, des expérimentations et des extensions des algorithmes développés tout au long du texte.

1. **(10%)** Étudiez l'influence de l'angle d'inclinaison dans l'étape de *deskew*. Générez des versions pivotées de l'image `skimage.data.page()` pour des angles entre $-10^\circ$ et $10^\circ$, appliquez l'algorithme présenté dans le chapitre et comparez l'angle estimé avec l'angle utilisé lors de la rotation. Présentez les résultats dans un tableau et discutez de la précision de la méthode.

2. **(15%)** Appliquez une normalisation du fond et un CLAHE (en utilisant au moins trois combinaisons de `clipLimit` et `tileGridSize`) sur l'image `skimage.data.page()` dégradée artificiellement avec un gradient d'éclairage et une ombre latérale. Segmentez chaque version à l'aide de la méthode d'Otsu et comparez les résultats en utilisant le nombre de composantes connexes parasites et la métrique IoU par rapport à un masque de référence construit manuellement.

3. **(15%)** Étudiez la sensibilité du filtrage par circularité, $C=\frac{4\pi A}{P^2},$ dans la détection des marqueurs circulaires. À l'aide du simulateur de la [Figure 6.9](#fig-06-sim-06-circularidade), générez des disques synthétiques avec un bruit géométrique croissant et évaluez les seuils $C\in\{0{,}5,\ 0{,}6,\ 0{,}7,\ 0{,}8\}$. Présentez un tableau reliant le seuil au nombre de faux positifs et de faux négatifs et discutez du compromis entre sensibilité et spécificité.

4. **(15%)** À partir des quatre marqueurs détectés, implémentez la rectification par perspective à l'aide de `cv2.getPerspectiveTransform` et `cv2.warpPerspective`. Ensuite, perturbez artificiellement les coordonnées des points de contrôle avec un bruit gaussien d'écart-type $\sigma\in\{1,3,5\}$ pixels et évaluez l'erreur de reprojection obtenue après l'homographie inverse.

5. **(15%)** Adaptez le *pipeline* d'acquisition et de rectification développé dans ce chapitre pour traiter des documents contenant des codes-barres linéaires en remplacement des *QRCodes*. Rasterisez le PDF avec `pdf2image` à 300 DPI, appliquez le *deskew*, décodez le symbole avec `pyzbar` et présentez l'image rectifiée accompagnée de la séquence de caractères obtenue.

6. **(15%)** Étendez la lecture des bulles du MCTest pour identifier trois situations : **OK**, **BLANC** (aucune alternative cochée) et **DOUBLE MARQUAGE** (deux alternatives ou plus au-dessus d'un seuil de remplissage). Évaluez au moins trois valeurs de ce seuil, présentez les résultats dans un `pandas.DataFrame` et discutez de son influence sur la classification des réponses.

7. **(15%)** Construisez un *pipeline* d'inspection industrielle combinant la soustraction d'images et l'analyse de variance locale de la texture sur un ensemble d'images synthétiques contenant des défauts simulés. Pour chaque image, générez un masque de référence (*ground truth*), calculez la métrique IoU (*Intersection over Union*) des deux approches pour différents seuils de décision et présentez les résultats dans des tableaux et des visualisations produites avec `mm.show`.

8. **(Bonus – 10%)** Implémentez manuellement l'estimation de l'angle d'inclinaison sans utiliser `cv2.HoughLines` ou `cv2.HoughLinesP`. À partir de la carte des contours obtenue par le détecteur de Canny, construisez l'accumulateur de la Transformée de Hough, $\rho=x\cos\theta+y\sin\theta,$ pour $\theta\in[-45^\circ,45^\circ]$, identifiez les maxima de l'accumulateur et estimez l'inclinaison par la médiane des droites détectées. Comparez les résultats avec l'implémentation d'OpenCV et discutez de l'influence des droites parasites sur l'estimation finale.

## Références du chapitre

Le fondement théorique et les études de cas présentés dans ce chapitre s’appuient sur les références suivantes :

- Gonzalez (2018), pour les fondements de la détection de contours, du seuillage, de la segmentation, des opérations morphologiques, de la reconnaissance optique de caractères et des transformations géométriques appliquées à l’analyse de documents.

- Szeliski (2022), pour la transformée de Hough, le recalage et l’alignement d’images, les transformations projectives (homographies) et les principes de l’inspection visuelle automatisée.

- Bradski (2008), pour l’utilisation de la bibliothèque OpenCV dans les étapes de détection de contours, de transformée de Hough, de transformations géométriques, d’analyse de contours et de décodage de *QR codes*.

- Smith (2007) et Smith (2013), pour l’architecture, le fonctionnement et l’évolution du moteur de reconnaissance optique de caractères **Tesseract OCR**, utilisé dans les exemples d’OCR présentés dans ce chapitre.

- Bahdanau (2015) et Vaswani (2017), pour les principes de la traduction automatique fondée sur les réseaux de neurones, y compris les mécanismes d’attention et les architectures *transformer*.

- Zampirolli (2023), pour la description du système MCTest, utilisé comme étude de cas d’un *pipeline* complet pour la lecture et la correction automatisées de feuilles de réponses.

- Bergmann (2019), Tabernik (2020) et Song (2013), pour les *jeux de données* publics d’inspection industrielle **MVTec AD**, **KolektorSDD** et **NEU Surface Defect Database**, utilisés comme référence pour l’évaluation et la comparaison des algorithmes de détection de défauts.

## Références du Chapitre


BAHDANAU, D.; CHO, K.; BENGIO, Y. **Neural Machine Translation by Jointly Learning to Align and Translate**. 2015.

BERGMANN, P. *et al*. **MVTec AD -- A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection**. 2019.

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SMITH, R. **An Overview of the Tesseract OCR Engine**. IEEE Computer Society, 2007.

SMITH, R. **History of the Tesseract OCR Engine: What Worked and What Didn't**. 2013.

SONG, K.; YAN, Y. **A Noise Robust Method Based on Completed Local Binary Patterns for Hot-Rolled Steel Strip Surface Defects**. 2013.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

TABERNIK, D. *et al*. **Segmentation-Based Deep-Learning Approach for Surface-Defect Detection**. 2020.

VASWANI, A. *et al*. **Attention Is All You Need**. 2017.

ZAMPIROLLI, F. A. **MCTest: Como Criar e Corrigir Exames Parametrizados**. Independente, 2023.